In [0]:
%pip install python-pptx openpyxl matplotlib pytz pandas numpy dask[dataframe]

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# ============================================================
# CONFIGURATION — Volumes I/O
# ============================================================
import os, glob
from datetime import datetime

VOLUME_CONFIG_PATH = '/Volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/narrative_deck_infigen/narrative_deck_config_final.xlsx'
VOLUME_TEMPLATE_PATH = '/Volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/narrative_deck_infigen/Narrative_Deck_final_template.pptx'

# Output goes to a separate volume
output_volume_dir = "/Volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/narrative_deck_output_path"

# ── Clean up old output decks ──
old_decks = glob.glob(os.path.join(output_volume_dir, "NARRATIVE_DECK_UPDATED_*.pptx"))
removed = 0
for f in old_decks:
    os.remove(f)
    removed += 1
print(f"🗑️  Removed {removed} old deck file(s) from output volume")

# ── Date-stamped output filename ──
today = datetime.now().strftime('%Y-%m-%d')
VOLUME_OUTPUT_PATH = os.path.join(output_volume_dir, f"NARRATIVE_DECK_UPDATED_{today}.pptx")

print("Config   :", VOLUME_CONFIG_PATH)
print("Template :", VOLUME_TEMPLATE_PATH)
print("Output   :", VOLUME_OUTPUT_PATH)

🗑️  Removed 1 old deck file(s) from output volume
Config   : /Volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/narrative_deck_infigen/narrative_deck_config_final.xlsx
Template : /Volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/narrative_deck_infigen/Narrative_Deck_final_template.pptx
Output   : /Volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/narrative_deck_output_path/NARRATIVE_DECK_UPDATED_2026-11-18.pptx


In [0]:
# No token needed — using spark.sql() for data fetching
# Disable ANSI mode so UNION ALL queries with mixed string/numeric types
# behave like the old SQL connector (returns NULL instead of throwing)
spark.conf.set("spark.sql.ansi.enabled", "false")
print("ANSI mode disabled for spark.sql() compatibility")

ANSI mode disabled for spark.sql() compatibility


In [0]:
import pandas as pd
import openpyxl
from pptx import Presentation
from io import BytesIO
import re
from pptx.chart.data import CategoryChartData
from lxml import etree
from pptx.util import Pt, Cm

In [0]:
# """
# Deck Update Pipeline + Highlight Application (Volumes version)
# --------------------------------------------------------------
# Converted from S3/SQL-connector notebook to Volumes-based output.
# - Uses spark.sql() instead of databricks SQL connector
# - Synchronous execution (no async/await)
# - Outputs to Databricks Volumes
# - Returns path via dbutils.notebook.exit()

# Two entry points:
#   run_deck_update()                    — Data updates only
#   run_deck_update_with_highlights()    — Data updates + highlights + speaker notes
# """

# import pandas as pd
# from pptx import Presentation
# from pptx.util import Pt, Cm
# from pptx.dml.color import RGBColor
# from pptx.enum.text import PP_ALIGN
# from pptx.enum.shapes import MSO_SHAPE
# from lxml import etree
# import re
# import json
# import math
# import os
# import shutil
# import zipfile
# import openpyxl
# import traceback
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from datetime import datetime
# from decimal import Decimal


# # ============================================================
# # XML NAMESPACES
# # ============================================================
# CHART_NS = {
#     'c': 'http://schemas.openxmlformats.org/drawingml/2006/chart',
#     'a': 'http://schemas.openxmlformats.org/drawingml/2006/main',
#     'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
# }
# NS_C = 'http://schemas.openxmlformats.org/drawingml/2006/chart'
# NS_A = 'http://schemas.openxmlformats.org/drawingml/2006/main'


# # ============================================================
# # 1. CONFIG READER
# # ============================================================
# def read_config(config_path: str) -> pd.DataFrame:
#     df = pd.read_excel(config_path)
#     print(f"Config loaded: {len(df)} elements to update")
#     print(df[['slide_num', 'shape_name', 'element_type']].to_string())
#     return df


# # ============================================================
# # 2. DATA FETCHER  (spark.sql — no token needed)
# # ============================================================
# def fetch_data(query_name, sql_query):
#     """Execute a SQL query via spark.sql and return a pandas DataFrame."""
#     print(f"  -> Fetching: {query_name}")
#     try:
#         df = spark.sql(sql_query).toPandas()
#         if 'dsg_deck' in df.columns and 'json_format_df' in df.columns:
#             if len(df) > 0:
#                 df = pd.read_json(df.iloc[0]['json_format_df'], orient='records')
#         print(f"     {len(df)} rows for {query_name}")
#         return df
#     except Exception as e:
#         print(f"     ERROR fetching {query_name}: {e}")
#         raise


# def fetch_all_data(config_df, max_parallel=15):
#     """Fetch all SQL data in parallel using ThreadPoolExecutor."""
#     data_cache = {}
#     sql_rows = config_df[config_df['sql_query'].notna() & (config_df['sql_query'].str.strip() != '')]
#     unique_sqls = sql_rows['sql_query'].drop_duplicates().tolist()
#     if not unique_sqls:
#         return data_cache

#     print(f"\nExecuting {len(unique_sqls)} SQL queries (max {max_parallel} concurrent)...")

#     sql_to_df = {}
#     with ThreadPoolExecutor(max_workers=max_parallel, thread_name_prefix='sql_fetch') as pool:
#         futures = {
#             pool.submit(fetch_data, f"SQL_{i+1}", sq): sq
#             for i, sq in enumerate(unique_sqls)
#         }
#         for future in as_completed(futures):
#             sq = futures[future]
#             try:
#                 sql_to_df[sq] = future.result()
#             except Exception as e:
#                 print(f"  Query failed: {e}")

#     for idx, row in config_df.iterrows():
#         sq = row.get('sql_query', '')
#         data_cache[idx] = sql_to_df.get(sq, pd.DataFrame()) if pd.notna(sq) and str(sq).strip() else pd.DataFrame()

#     print(f"Data cache built: {len(data_cache)} datasets")
#     return data_cache


# # ============================================================
# # 3. SHAPE FINDER
# # ============================================================
# def find_shape(prs, slide_num, shape_name):
#     slide_index = slide_num - 1
#     if slide_index < 0 or slide_index >= len(prs.slides):
#         return None
#     for shape in prs.slides[slide_index].shapes:
#         if shape.name == shape_name:
#             return shape
#     print(f"  Shape '{shape_name}' not found on slide {slide_num}")
#     return None


# # ============================================================
# # 4. DATA TYPE CONVERTERS
# # ============================================================
# def date_string_to_excel_serial(date_str):
#     for fmt in ['%b-%y', '%B-%y', '%b-%Y', '%Y-%m-%d', '%m/%d/%Y']:
#         try:
#             dt = datetime.strptime(date_str, fmt)
#             return float((dt - datetime(1899, 12, 30)).days)
#         except ValueError:
#             continue
#     try:
#         return float(date_str)
#     except (ValueError, TypeError):
#         return date_str

# def is_null_value(val):
#     if val is None: return True
#     if isinstance(val, float) and math.isnan(val): return True
#     if isinstance(val, str) and val.strip() == '': return True
#     return False

# def convert_to_float(val):
#     if is_null_value(val): return None
#     if isinstance(val, Decimal): return float(val)
#     if isinstance(val, (int, float)): return float(val)
#     try: return float(val)
#     except: return None

# def detect_category_type(chart):
#     tree = chart._chartSpace
#     ser_elements = tree.findall('.//c:ser', CHART_NS)
#     if ser_elements:
#         cat_el = ser_elements[0].find('c:cat', CHART_NS)
#         if cat_el is not None:
#             if cat_el.find('c:numRef', CHART_NS) is not None: return 'numeric'
#             if cat_el.find('c:strRef', CHART_NS) is not None: return 'string'
#     return 'string'

# def prepare_chart_data(chart, df):
#     cat_type = detect_category_type(chart)
#     raw_cats = df.iloc[:, 0].tolist()
#     if cat_type == 'numeric':
#         categories = []
#         for v in raw_cats:
#             if is_null_value(v): categories.append(None)
#             elif isinstance(v, str): categories.append(date_string_to_excel_serial(v))
#             elif isinstance(v, (int, float, Decimal)): categories.append(float(v))
#             else: categories.append(v)
#     else:
#         categories = [str(v) if not is_null_value(v) else None for v in raw_cats]
#     series_data = [[convert_to_float(v) for v in df.iloc[:, c].tolist()] for c in range(1, len(df.columns))]
#     return categories, series_data


# # ============================================================
# # 5. CHART UPDATER
# # ============================================================
# def update_chart_data(chart, df):
#     tree = chart._chartSpace
#     categories, series_data = prepare_chart_data(chart, df)
#     for cat_el in tree.findall('.//c:cat', CHART_NS):
#         _update_ref_values(cat_el, categories)
#     for i, ser_el in enumerate(tree.findall('.//c:ser', CHART_NS)):
#         if i < len(series_data):
#             val_el = ser_el.find('c:val', CHART_NS)
#             if val_el is not None:
#                 _update_ref_values(val_el, series_data[i])
#     print(f"  Chart updated: {len(categories)} cats x {len(series_data)} series")

# def _update_ref_values(parent_el, values):
#     for ref_type in ['c:numRef', 'c:strRef']:
#         ref_el = parent_el.find(ref_type, CHART_NS)
#         if ref_el is not None:
#             f_el = ref_el.find('c:f', CHART_NS)
#             if f_el is not None and f_el.text:
#                 f_el.text = _adjust_formula_range(f_el.text, len(values))
#             cache_tag = 'c:numCache' if 'num' in ref_type else 'c:strCache'
#             cache_el = ref_el.find(cache_tag, CHART_NS)
#             if cache_el is not None:
#                 _replace_cache_values(cache_el, values)
#                 return
#     for lit_type in ['c:numLit', 'c:strLit']:
#         lit_el = parent_el.find(lit_type, CHART_NS)
#         if lit_el is not None:
#             _replace_cache_values(lit_el, values)
#             return

# def _adjust_formula_range(formula, data_len):
#     m = re.match(r"(.+\$[A-Z]+\$)(\d+)(:\$[A-Z]+\$)(\d+)$", formula)
#     if m:
#         return f"{m.group(1)}{m.group(2)}{m.group(3)}{int(m.group(2)) + data_len - 1}"
#     m2 = re.match(r"(.+\$)(\d+)$", formula)
#     if m2:
#         sm = re.search(r'\$(\d+):', formula)
#         sr = int(sm.group(1)) if sm else 2
#         return f"{m2.group(1)}{sr + data_len - 1}"
#     return formula

# def _replace_cache_values(cache_el, values):
#     ns_c = CHART_NS['c']
#     pt_count = cache_el.find('c:ptCount', CHART_NS)
#     if pt_count is not None:
#         pt_count.set('val', str(len(values)))
#     for pt in cache_el.findall('c:pt', CHART_NS):
#         cache_el.remove(pt)
#     for idx, val in enumerate(values):
#         if val is None: continue
#         pt = etree.SubElement(cache_el, f'{{{ns_c}}}pt')
#         pt.set('idx', str(idx))
#         v = etree.SubElement(pt, f'{{{ns_c}}}v')
#         v.text = repr(val) if isinstance(val, float) else str(val)


# # ============================================================
# # 5b. LAST-POINT DATA LABELS
# # ============================================================
# def add_last_point_data_labels(chart, df):
#     for plot in chart.plots:
#         for series in plot.series:
#             ser_el = series._element
#             is_dashed = False
#             spPr = ser_el.find(f'{{{NS_C}}}spPr')
#             if spPr is not None:
#                 ln = spPr.find(f'{{{NS_A}}}ln')
#                 if ln is not None:
#                     pd_el = ln.find(f'{{{NS_A}}}prstDash')
#                     if pd_el is not None and pd_el.get('val', '') not in ('', 'solid'): is_dashed = True
#                     if ln.find(f'{{{NS_A}}}custDash') is not None: is_dashed = True
#             if is_dashed: continue
#             hex_color = None
#             if spPr is not None:
#                 ln = spPr.find(f'{{{NS_A}}}ln')
#                 if ln is not None:
#                     sf = ln.find(f'{{{NS_A}}}solidFill')
#                     if sf is not None:
#                         sc = sf.find(f'{{{NS_A}}}srgbClr')
#                         if sc is not None: hex_color = sc.get('val')
#             last_idx = 0
#             for vt in ['val', 'yVal']:
#                 ve = ser_el.find(f'{{{NS_C}}}{vt}')
#                 if ve is not None:
#                     for cp in [f'{{{NS_C}}}numRef/{{{NS_C}}}numCache', f'{{{NS_C}}}numLit']:
#                         cache = ve.find(cp)
#                         if cache is not None:
#                             pts = cache.findall(f'{{{NS_C}}}pt')
#                             if pts: last_idx = max(int(p.get('idx', '0')) for p in pts)
#                     break
#             old = ser_el.find(f'{{{NS_C}}}dLbls')
#             if old is not None: ser_el.remove(old)
#             dLbls = etree.Element(f'{{{NS_C}}}dLbls')
#             ib = None
#             for tag in ['cat', 'val', 'xVal', 'yVal', 'bubbleSize', 'smooth', 'extLst']:
#                 el = ser_el.find(f'{{{NS_C}}}{tag}')
#                 if el is not None: ib = el; break
#             if ib is not None: ser_el.insert(list(ser_el).index(ib), dLbls)
#             else: ser_el.append(dLbls)
#             dLbl = etree.SubElement(dLbls, f'{{{NS_C}}}dLbl')
#             etree.SubElement(dLbl, f'{{{NS_C}}}idx').set('val', str(last_idx))
#             nf = etree.SubElement(dLbl, f'{{{NS_C}}}numFmt')
#             nf.set('formatCode', '0%'); nf.set('sourceLinked', '0')
#             etree.SubElement(dLbl, f'{{{NS_C}}}dLblPos').set('val', 'r')
#             txPr = etree.SubElement(dLbl, f'{{{NS_C}}}txPr')
#             etree.SubElement(txPr, f'{{{NS_A}}}bodyPr')
#             etree.SubElement(txPr, f'{{{NS_A}}}lstStyle')
#             p = etree.SubElement(txPr, f'{{{NS_A}}}p')
#             pPr = etree.SubElement(p, f'{{{NS_A}}}pPr')
#             rPr = etree.SubElement(pPr, f'{{{NS_A}}}defRPr')
#             rPr.set('b', '1'); rPr.set('sz', '900'); rPr.set('lang', 'en-US')
#             if hex_color:
#                 sf2 = etree.SubElement(rPr, f'{{{NS_A}}}solidFill')
#                 etree.SubElement(sf2, f'{{{NS_A}}}srgbClr').set('val', hex_color)
#             etree.SubElement(p, f'{{{NS_A}}}endParaRPr').set('lang', 'en-US')
#             for tag, val in [('showLegendKey','0'),('showVal','1'),('showCatName','0'),
#                              ('showSerName','0'),('showPercent','0'),('showBubbleSize','0')]:
#                 etree.SubElement(dLbl, f'{{{NS_C}}}{tag}').set('val', val)
#             nf2 = etree.SubElement(dLbls, f'{{{NS_C}}}numFmt')
#             nf2.set('formatCode', '0%'); nf2.set('sourceLinked', '0')
#             for tag in ['showLegendKey','showVal','showCatName','showSerName','showPercent','showBubbleSize']:
#                 etree.SubElement(dLbls, f'{{{NS_C}}}{tag}').set('val', '0')
#             print(f"    Label idx={last_idx}, color=#{hex_color}")
#     print(f"  Labels applied")


# # ============================================================
# # 6. TEXTBOX UPDATER
# # ============================================================
# def update_textbox(shape, df, static_text=None):
#     if static_text and str(static_text).strip() and not pd.isna(static_text):
#         display_text = str(static_text)
#     elif not df.empty:
#         val = df.iloc[0, 0]
#         if isinstance(val, Decimal): val = float(val)
#         display_text = str(val)
#     else:
#         return False
#     if not shape.has_text_frame: return False
#     tf = shape.text_frame
#     for para in tf.paragraphs:
#         if para.runs:
#             para.runs[0].text = display_text
#             for run in para.runs[1:]: run.text = ''
#             # Triangle color coding
#             if '▼' in display_text or '▾' in display_text:
#                 para.runs[0].font.color.rgb = RGBColor(0xFF, 0x00, 0x00)
#             elif '▲' in display_text or '▴' in display_text:
#                 para.runs[0].font.color.rgb = RGBColor(0x00, 0x80, 0x00)
#             break
#     print(f"  Textbox updated: '{display_text}'")
#     return True


# # ============================================================
# # 6b. EXEC SUMMARY TEXTBOX UPDATER
# # ============================================================
# def update_exec_summary_textbox(shape, text, title_font_size=9, bullet_font_size=8):
#     if not shape.has_text_frame:
#         print(f"  Shape has no text frame")
#         return False

#     tf = shape.text_frame
#     NS_A_URI = 'http://schemas.openxmlformats.org/drawingml/2006/main'

#     template_font_name = 'Calibri'
#     template_font_color = None
#     if tf.paragraphs and tf.paragraphs[0].runs:
#         tmpl_run = tf.paragraphs[0].runs[0]
#         if tmpl_run.font.name:
#             template_font_name = tmpl_run.font.name
#         try:
#             if tmpl_run.font.color and tmpl_run.font.color.type is not None:
#                 template_font_color = str(tmpl_run.font.color.rgb)
#         except (AttributeError, TypeError):
#             template_font_color = None

#     p_elements = list(tf._txBody.findall(f'{{{NS_A_URI}}}p'))
#     for p_el in p_elements:
#         tf._txBody.remove(p_el)

#     lines = text.split('\n')
#     content_lines = [l for l in lines if l.strip()]

#     if not content_lines:
#         print(f"  No content lines to write")
#         return False

#     for i, line in enumerate(content_lines):
#         stripped = line.strip()
#         is_bullet = stripped.startswith('- ')

#         p_el = etree.SubElement(tf._txBody, f'{{{NS_A_URI}}}p')
#         pPr = etree.SubElement(p_el, f'{{{NS_A_URI}}}pPr')

#         if is_bullet:
#             pPr.set('marL', '180000')
#             pPr.set('indent', '-90000')
#             spc_before = etree.SubElement(pPr, f'{{{NS_A_URI}}}spcBef')
#             spc_pts = etree.SubElement(spc_before, f'{{{NS_A_URI}}}spcPts')
#             spc_pts.set('val', '100')

#             r = etree.SubElement(p_el, f'{{{NS_A_URI}}}r')
#             rPr = etree.SubElement(r, f'{{{NS_A_URI}}}rPr')
#             rPr.set('lang', 'en-US')
#             rPr.set('sz', str(bullet_font_size * 100))
#             rPr.set('b', '0')
#             rPr.set('dirty', '0')

#             latin = etree.SubElement(rPr, f'{{{NS_A_URI}}}latin')
#             latin.set('typeface', template_font_name)

#             if template_font_color:
#                 solidFill = etree.SubElement(rPr, f'{{{NS_A_URI}}}solidFill')
#                 srgbClr = etree.SubElement(solidFill, f'{{{NS_A_URI}}}srgbClr')
#                 srgbClr.set('val', template_font_color)

#             t = etree.SubElement(r, f'{{{NS_A_URI}}}t')
#             t.text = stripped
#         else:
#             if i > 0:
#                 spc_before = etree.SubElement(pPr, f'{{{NS_A_URI}}}spcBef')
#                 spc_pts = etree.SubElement(spc_before, f'{{{NS_A_URI}}}spcPts')
#                 spc_pts.set('val', '600')

#             r = etree.SubElement(p_el, f'{{{NS_A_URI}}}r')
#             rPr = etree.SubElement(r, f'{{{NS_A_URI}}}rPr')
#             rPr.set('lang', 'en-US')
#             rPr.set('sz', str(title_font_size * 100))
#             rPr.set('b', '1')
#             rPr.set('dirty', '0')

#             latin = etree.SubElement(rPr, f'{{{NS_A_URI}}}latin')
#             latin.set('typeface', template_font_name)

#             if template_font_color:
#                 solidFill = etree.SubElement(rPr, f'{{{NS_A_URI}}}solidFill')
#                 srgbClr = etree.SubElement(solidFill, f'{{{NS_A_URI}}}srgbClr')
#                 srgbClr.set('val', template_font_color)

#             t = etree.SubElement(r, f'{{{NS_A_URI}}}t')
#             t.text = stripped

#     title_count = len([l for l in content_lines if not l.strip().startswith('- ')])
#     bullet_count = len([l for l in content_lines if l.strip().startswith('- ')])
#     print(f"  Exec summary updated: {len(content_lines)} lines "
#           f"({title_count} titles bold, {bullet_count} bullets indented)")
#     return True


# # ============================================================
# # 7. TABLE / EXCEL UPDATER
# # ============================================================
# def update_table(shape, df):
#     if not shape.has_table: return False
#     table = shape.table
#     for r_idx, row_data in enumerate(df.values):
#         if r_idx >= len(table.rows): break
#         for c_idx, val in enumerate(row_data):
#             if c_idx >= len(table.columns): break
#             cell = table.cell(r_idx, c_idx)
#             if is_null_value(val): val = ''
#             elif isinstance(val, Decimal): val = float(val)
#             if cell.text_frame.paragraphs and cell.text_frame.paragraphs[0].runs:
#                 cell.text_frame.paragraphs[0].runs[0].text = str(val)
#                 for run in cell.text_frame.paragraphs[0].runs[1:]: run.text = ''
#             else:
#                 cell.text = str(val)
#     print(f"  Table updated: {len(df)}r x {len(df.columns)}c")
#     return True

# def get_chart_excel_mapping(pptx_path):
#     mapping = {}
#     with zipfile.ZipFile(pptx_path, 'r') as zf:
#         for name in zf.namelist():
#             if name.startswith('ppt/charts/_rels/') and name.endswith('.rels'):
#                 chart_name = name.split('/')[-1].replace('.rels', '')
#                 rels_xml = zf.read(name)
#                 for rel in etree.fromstring(rels_xml).findall(
#                     './/{http://schemas.openxmlformats.org/package/2006/relationships}Relationship'):
#                     target = rel.get('Target')
#                     if target and 'embeddings' in target:
#                         mapping[chart_name] = target.split('/')[-1]
#     return mapping

# def update_embedded_excel_in_pptx(pptx_path, output_path, chart_excel_data):
#     temp_dir = 'temp_pptx_extract'
#     if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
#     with zipfile.ZipFile(pptx_path, 'r') as zf: zf.extractall(temp_dir)
#     embeddings_dir = os.path.join(temp_dir, 'ppt', 'embeddings')
#     if os.path.exists(embeddings_dir):
#         for excel_name, df in chart_excel_data.items():
#             excel_path = os.path.join(embeddings_dir, excel_name)
#             if not os.path.exists(excel_path) or df.empty: continue
#             try:
#                 wb = openpyxl.load_workbook(excel_path)
#                 ws = wb.active
#                 merged = set()
#                 for mr in ws.merged_cells.ranges:
#                     for r in range(mr.min_row, mr.max_row+1):
#                         for c in range(mr.min_col, mr.max_col+1):
#                             if not (r == mr.min_row and c == mr.min_col): merged.add((r, c))
#                 for r in range(2, ws.max_row+1):
#                     for c in range(1, ws.max_column+1):
#                         if (r, c) not in merged:
#                             try: ws.cell(row=r, column=c, value=None)
#                             except: pass
#                 for r_idx, row_data in enumerate(df.values):
#                     for c_idx, val in enumerate(row_data):
#                         tr, tc = 2+r_idx, 1+c_idx
#                         if (tr, tc) in merged: continue
#                         if is_null_value(val): val = None
#                         elif isinstance(val, Decimal): val = float(val)
#                         elif isinstance(val, str):
#                             for fmt in ['%b-%y','%B-%y','%Y-%m-%d','%b-%Y']:
#                                 try: val = datetime.strptime(val, fmt); break
#                                 except: continue
#                         try: ws.cell(row=tr, column=tc, value=val)
#                         except: continue
#                 wb.save(excel_path)
#                 print(f"    Updated: {excel_name}")
#             except Exception as e:
#                 print(f"    Failed: {excel_name} - {e}")
#     with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zf:
#         for root, dirs, files in os.walk(temp_dir):
#             for f in files:
#                 fp = os.path.join(root, f)
#                 zf.write(fp, os.path.relpath(fp, temp_dir))
#     shutil.rmtree(temp_dir)


# # ============================================================
# # 8. ELEMENT UPDATER
# # ============================================================
# def update_element(prs, slide_num, shape_name, element_type, df, static_text=None, show_last_label=False):
#     shape = find_shape(prs, slide_num, shape_name)
#     if shape is None: return False, None
#     if element_type == 'chart':
#         if df.empty or not shape.has_chart: return False, None
#         chart = shape.chart
#         chart_xml_name = chart.part.partname.split('/')[-1]
#         update_chart_data(chart, df)
#         if show_last_label: add_last_point_data_labels(chart, df)
#         return True, chart_xml_name
#     elif element_type == 'textbox':
#         return update_textbox(shape, df, static_text), None
#     elif element_type == 'exec_summary':
#         display_text = None
#         if static_text and str(static_text).strip() and not pd.isna(static_text):
#             display_text = str(static_text)
#         elif not df.empty:
#             val = df.iloc[0, 0]
#             if isinstance(val, Decimal): val = float(val)
#             display_text = str(val)
#         if not display_text or display_text.strip() == '' or display_text == 'nan':
#             print(f"  No exec summary text to update")
#             return False, None
#         ok = update_exec_summary_textbox(shape, display_text)
#         return ok, None
#     elif element_type == 'table':
#         if df.empty or not shape.has_table: return False, None
#         return update_table(shape, df), None
#     return False, None


# # ============================================================
# # 9. MAIN ORCHESTRATOR (synchronous, Volumes output)
# # ============================================================
# def run_deck_update(config_path, template_path, output_path,
#                     max_parallel_queries=15, update_embedded_excel=True):
#     print("=" * 60 + "\nDECK UPDATE STARTED\n" + "=" * 60)
#     config_df = read_config(config_path)
#     data_cache = fetch_all_data(config_df, max_parallel_queries)
#     prs = Presentation(template_path)
#     print(f"\nTemplate: {len(prs.slides)} slides")
#     chart_excel_mapping = get_chart_excel_mapping(template_path) if update_embedded_excel else {}
#     success, fail, chart_data_for_excel, chart_data_map = 0, 0, {}, {}

#     for idx, row in config_df.iterrows():
#         slide_num = int(row['slide_num'])
#         try:
#             static_text = row.get('static_text', None)
#             if pd.isna(static_text): static_text = None
#             slr = row.get('show_last_label', False)
#             show_last_label = False if pd.isna(slr) else str(slr).strip().upper() in ('TRUE','1','1.0','YES')
#             df = data_cache.get(idx, pd.DataFrame())
#             ok, chart_xml_name = debug_wrap_update(
#                                     prs, slide_num, row['shape_name'], row['element_type'],
#                                     df, static_text, show_last_label
#                                 )
#             if ok:
#                 success += 1
#                 if row['element_type'] == 'chart' and not df.empty:
#                     chart_data_map[(slide_num, row['shape_name'])] = df
#                 if update_embedded_excel and chart_xml_name and chart_xml_name in chart_excel_mapping:
#                     chart_data_for_excel[chart_excel_mapping[chart_xml_name]] = df
#             else:
#                 fail += 1
#         except Exception as e:
#             print(f"  Error: {e}")
#             traceback.print_exc()
#             fail += 1

#     # Save to LOCAL temp path first (Volumes FUSE doesn't support zipfile close)
#     local_temp = "/tmp/_deck_temp.pptx"
#     local_final = "/tmp/_deck_final.pptx"
#     prs.save(local_temp)
#     if update_embedded_excel and chart_data_for_excel:
#         update_embedded_excel_in_pptx(local_temp, local_final, chart_data_for_excel)
#         os.remove(local_temp)
#     else:
#         shutil.move(local_temp, local_final)

#     # Copy bytes to Volumes
#     with open(local_final, 'rb') as f:
#         pptx_bytes = f.read()
#     with open(output_path, 'wb') as f:
#         f.write(pptx_bytes)
#     os.remove(local_final)
#     print(f"\nSuccess: {success}, Failed: {fail}, Saved to: {output_path}")
#     return chart_data_map


# # ################################################################
# # ##  HIGHLIGHT APPLICATION — APPENDED BELOW (core untouched)   ##
# # ################################################################

# HIGHLIGHT_HEX = '000000'
# COMBINED_TABLE = "`commercial-us-apo-onc-iia-dev`.`infigen_adhocs`.claims_deck_insights_highlights"


# def hl_detect_chart_type(chart):
#     tree = chart._chartSpace
#     pa = tree.find('.//c:plotArea', CHART_NS)
#     if pa is None: return 'unknown'
#     for child in pa:
#         tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
#         if tag in ('lineChart', 'line3DChart'): return 'line'
#         if tag in ('barChart', 'bar3DChart'):
#             bd = child.find('c:barDir', CHART_NS)
#             return 'horizontal_bar' if bd is not None and bd.get('val') == 'bar' else 'stacked_bar'
#         if tag in ('areaChart', 'area3DChart'): return 'area'
#         if tag in ('pieChart', 'pie3DChart', 'doughnutChart'): return 'pie'
#         if tag == 'scatterChart': return 'scatter'
#     return 'unknown'


# def hl_find_category_index_from_df(data_df, target_category):
#     if data_df is None or data_df.empty: return None
#     target = target_category.strip()
#     for i, cat in enumerate(data_df.iloc[:, 0].tolist()):
#         cs = str(cat).strip()
#         if cs == target or cs.startswith(target) or target in cs or cs in target:
#             return i
#     return None

# def hl_get_series_name(ser_el):
#     tx = ser_el.find('c:tx', CHART_NS)
#     if tx is not None:
#         for v in tx.iter():
#             if v.text and v.tag.endswith('}v'): return v.text.strip()
#     return None

# def hl_matches_series(a, b):
#     if not a or not b: return False
#     a, b = a.strip().upper(), b.strip().upper()
#     return a == b or a in b or b in a

# def hl_find_point_index_in_series(ser_el, cat_idx):
#     for vt in ['c:val', 'c:yVal']:
#         ve = ser_el.find(vt, CHART_NS)
#         if ve is not None:
#             for ct in ['c:numCache', 'c:numLit']:
#                 cache = ve.find(ct, CHART_NS)
#                 if cache is not None:
#                     indices = sorted(int(p.get('idx','0')) for p in cache.findall('c:pt', CHART_NS))
#                     if cat_idx in indices: return cat_idx
#                     valid = [i for i in indices if i <= cat_idx]
#                     return valid[-1] if valid else (indices[-1] if indices else cat_idx)
#             break
#     return cat_idx

# def hl_parse_value(val):
#     if val is None: return None
#     if isinstance(val, Decimal): return float(val)
#     if isinstance(val, (int, float)):
#         return None if math.isnan(val) else float(val)
#     if isinstance(val, str):
#         val = val.strip()
#         if val in ('', 'nan', 'none', 'None'): return None
#         if '%' in val:
#             try: return float(val.replace('%','').strip()) / 100.0
#             except: return None
#         try: return float(val)
#         except: return None
#     return None


# def hl_read_plot_area(chart, chart_shape):
#     tree = chart._chartSpace
#     sl, st, sw, sh = chart_shape.left, chart_shape.top, chart_shape.width, chart_shape.height
#     pa = tree.find('.//c:plotArea', CHART_NS)
#     if pa is not None:
#         layout = pa.find('c:layout', CHART_NS)
#         if layout is not None:
#             ml = layout.find('c:manualLayout', CHART_NS)
#             if ml is not None:
#                 x, y, w, h = [ml.find(f'c:{t}', CHART_NS) for t in ['x','y','w','h']]
#                 if all(e is not None for e in [x, y, w, h]):
#                     try:
#                         xf, yf, wf, hf = float(x.get('val','0')), float(y.get('val','0')), float(w.get('val','1')), float(h.get('val','1'))
#                         pl = sl + int(xf * sw)
#                         pt = st + int(yf * sh)
#                         pw, ph = int(wf * sw), int(hf * sh)
#                         return {'left': pl, 'top': pt, 'right': pl+pw, 'bottom': pt+ph, 'width': pw, 'height': ph}
#                     except: pass
#     pl = sl + int(sw*0.08); pr = sl+sw - int(sw*0.03)
#     pt2 = st + int(sh*0.05); pb = st+sh - int(sh*0.15)
#     return {'left': pl, 'top': pt2, 'right': pr, 'bottom': pb, 'width': pr-pl, 'height': pb-pt2}

# def hl_read_gap_width(chart):
#     tree = chart._chartSpace
#     for ct in ['c:barChart', 'c:bar3DChart']:
#         ce = tree.find(f'.//{ct}', CHART_NS)
#         if ce is not None:
#             gw = ce.find('c:gapWidth', CHART_NS)
#             if gw is not None:
#                 try: return int(gw.get('val', '150'))
#                 except: pass
#     return 150


# def hl_calc_segment_rect(chart, chart_shape, data_df, cat_idx, target_series, direction='vertical'):
#     if data_df is None or data_df.empty: return None
#     plot = hl_read_plot_area(chart, chart_shape)
#     gap_pct = hl_read_gap_width(chart)
#     n_cats = data_df.shape[0]
#     series_names = list(data_df.columns[1:])

#     tree_cb = chart._chartSpace
#     cross_between = 'between'
#     for at in ['c:catAx', 'c:dateAx']:
#         for ax in tree_cb.findall(f'.//{at}', CHART_NS):
#             cb = ax.find('c:crossBetween', CHART_NS)
#             if cb is not None: cross_between = cb.get('val', 'between')

#     if direction == 'horizontal':
#         cat_span, val_span = plot['height'], plot['width']
#         cat_orientation = None
#         for at in ['c:catAx', 'c:dateAx']:
#             for ax in tree_cb.findall(f'.//{at}', CHART_NS):
#                 sc = ax.find('c:scaling', CHART_NS)
#                 if sc is not None:
#                     ori = sc.find('c:orientation', CHART_NS)
#                     if ori is not None:
#                         cat_orientation = ori.get('val', '')
#                         print(f"    [Rect] Found orientation={cat_orientation} on {at}")
#         if cat_orientation is None:
#             cat_orientation = 'minMax'
#             print(f"    [Rect] No catAx orientation found — defaulting to minMax (reversed)")
#         if cat_orientation == 'minMax':
#             pos_cat_idx = n_cats - 1 - cat_idx
#         else:
#             pos_cat_idx = cat_idx
#         print(f"    [Rect] horizontal: cat_idx={cat_idx}→pos={pos_cat_idx} (orientation={cat_orientation})")
#     else:
#         cat_span, val_span = plot['width'], plot['height']
#         pos_cat_idx = cat_idx

#     if cross_between == 'midCat' and n_cats > 1:
#         bp = cat_span / (n_cats - 1)
#         bt = int(bp / (1 + gap_pct/100.0))
#         cc = int(pos_cat_idx * bp)
#     else:
#         bp = cat_span / n_cats
#         bt = int(bp / (1 + gap_pct/100.0))
#         cc = int((pos_cat_idx + 0.5) * bp)
#     cs = cc - bt // 2
#     tol = int(bt * 0.04); cs -= tol; bt += tol * 2

#     axis_max = 1.0
#     for ax in tree_cb.findall('.//c:valAx', CHART_NS):
#         sc = ax.find('c:scaling', CHART_NS)
#         if sc is not None:
#             mx = sc.find('c:max', CHART_NS)
#             if mx is not None:
#                 try: axis_max = float(mx.get('val', '1'))
#                 except: pass

#     cum, seg_val, found = 0.0, 0.0, False
#     for col in series_names:
#         v = hl_parse_value(data_df.iloc[cat_idx][col])
#         if v is None: v = 0.0
#         if hl_matches_series(col, target_series):
#             seg_val = v; found = True; break
#         cum += v
#     if not found or seg_val == 0: return None

#     sf = cum / axis_max
#     ef = (cum + seg_val) / axis_max

#     if direction == 'horizontal':
#         return {'left': plot['left'] + int(sf * val_span), 'top': plot['top'] + cs,
#                 'width': int((ef - sf) * val_span), 'height': bt}
#     else:
#         st_y = plot['bottom'] - int(ef * val_span)
#         sb_y = plot['bottom'] - int(sf * val_span)
#         return {'left': plot['left'] + cs, 'top': min(st_y, sb_y),
#                 'width': bt, 'height': abs(sb_y - st_y)}


# def hl_calc_line_point_rect(chart, chart_shape, data_df, cat_idx, target_series):
#     if data_df is None or data_df.empty: return None
#     plot = hl_read_plot_area(chart, chart_shape)
#     n_cats = data_df.shape[0]
#     series_names = list(data_df.columns[1:])

#     x_step = plot['width'] / max(n_cats - 1, 1)
#     point_x = plot['left'] + int(cat_idx * x_step)

#     target_val = None
#     for col in series_names:
#         if hl_matches_series(col, target_series):
#             target_val = hl_parse_value(data_df.iloc[cat_idx][col])
#             break
#     if target_val is None: return None

#     tree = chart._chartSpace
#     axis_min = None
#     axis_max = None
#     for ax in tree.findall('.//c:valAx', CHART_NS):
#         sc = ax.find('c:scaling', CHART_NS)
#         if sc is not None:
#             mn = sc.find('c:min', CHART_NS)
#             mx = sc.find('c:max', CHART_NS)
#             if mn is not None:
#                 try: axis_min = float(mn.get('val', '0'))
#                 except: pass
#             if mx is not None:
#                 try: axis_max = float(mx.get('val', '1'))
#                 except: pass

#     if axis_max is None:
#         all_vals = []
#         for col in series_names:
#             for v in data_df[col].tolist():
#                 pv = hl_parse_value(v)
#                 if pv is not None: all_vals.append(pv)
#         if all_vals:
#             data_max = max(all_vals)
#             if data_max <= 1:
#                 axis_max = math.ceil(data_max * 20) / 20
#             else:
#                 axis_max = math.ceil(data_max * 10) / 10
#         else:
#             axis_max = 1.0
#     if axis_min is None:
#         axis_min = 0.0

#     value_range = axis_max - axis_min if (axis_max - axis_min) != 0 else 1
#     point_y = plot['bottom'] - int(((target_val - axis_min) / value_range) * plot['height'])

#     print(f"      [LinePos] series='{target_series}', val={target_val}, "
#           f"axis=[{axis_min}, {axis_max}], y={point_y}")

#     return {'left': point_x, 'top': point_y, 'width': 0, 'height': 0,
#             'center_x': point_x, 'center_y': point_y}


# def hl_add_rectangle_overlay(slide, rect, padding=Cm(0.1)):
#     shape = slide.shapes.add_shape(
#         MSO_SHAPE.RECTANGLE,
#         left=int(rect['left'] - padding), top=int(rect['top'] - padding),
#         width=int(rect['width'] + padding*2), height=int(rect['height'] + padding*2))
#     shape.fill.background()
#     shape.line.color.rgb = RGBColor(0, 0, 0)
#     shape.line.width = Pt(1.5)
#     ln_el = shape._element.find(f'.//{{{NS_A}}}ln')
#     if ln_el is not None:
#         etree.SubElement(ln_el, f'{{{NS_A}}}prstDash').set('val', 'dash')
#     shape.name = "HL_Rect"
#     return shape

# def hl_add_callout_textbox(slide, rect, callout_text, position='right', direction='vertical'):
#     box_w, box_h, gap = Cm(2.5), Cm(0.45), Cm(0.1)
#     if position == 'right':
#         bl = rect['left'] + rect['width'] + gap
#         bt = rect['top'] + rect['height']//2 - box_h//2
#     elif position == 'left':
#         bl = rect['left'] - box_w - gap
#         bt = rect['top'] + rect['height']//2 - box_h//2
#     elif position == 'top':
#         bl = rect['left'] + rect['width']//2 - box_w//2
#         bt = rect['top'] - box_h - gap
#     elif position == 'bottom':
#         bl = rect['left'] + rect['width']//2 - box_w//2
#         bt = rect['top'] + rect['height'] + gap
#     else:
#         if direction == 'horizontal':
#             bl = rect['left'] + rect['width']//2 - box_w//2
#             bt = rect['top'] + rect['height'] + gap
#         else:
#             bl = rect['left'] + rect['width'] + gap
#             bt = rect['top'] + rect['height']//2 - box_h//2
#     txBox = slide.shapes.add_textbox(left=int(bl), top=int(bt), width=int(box_w), height=int(box_h))
#     txBox.fill.solid()
#     txBox.fill.fore_color.rgb = RGBColor(0, 0, 0)
#     txBox.line.fill.background()
#     tf = txBox.text_frame
#     tf.word_wrap = True
#     tf.margin_top = tf.margin_bottom = Cm(0.03)
#     tf.margin_left = tf.margin_right = Cm(0.1)
#     p = tf.paragraphs[0]
#     p.alignment = PP_ALIGN.CENTER
#     run = p.add_run()
#     run.text = callout_text
#     run.font.size = Pt(7)
#     run.font.bold = True
#     run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
#     txBox.name = "HL_Callout"
#     return txBox


# def hl_restyle_line_labels(chart, highlight_spec, data_df=None):
#     highlights = highlight_spec.get('highlights', [])
#     tree = chart._chartSpace
#     for hl in highlights:
#         target_series = hl.get('series', '')
#         target_category = hl.get('category', '')
#         cat_idx = hl_find_category_index_from_df(data_df, target_category)
#         if cat_idx is None: continue
#         for ser_el in tree.findall('.//c:ser', CHART_NS):
#             sn = hl_get_series_name(ser_el)
#             if not hl_matches_series(sn, target_series): continue
#             point_idx = hl_find_point_index_in_series(ser_el, cat_idx)
#             print(f"    Line highlight: '{sn}' idx={point_idx}")
#             dLbls = ser_el.find(f'{{{NS_C}}}dLbls')
#             if dLbls is not None:
#                 for dLbl in dLbls.findall(f'{{{NS_C}}}dLbl'):
#                     idx_el = dLbl.find(f'{{{NS_C}}}idx')
#                     if idx_el is not None and idx_el.get('val') == str(point_idx):
#                         txPr = dLbl.find(f'{{{NS_C}}}txPr')
#                         if txPr is not None:
#                             for defRPr in txPr.iter():
#                                 if defRPr.tag == f'{{{NS_A}}}defRPr':
#                                     defRPr.set('sz', '1000')
#                                     old_fill = defRPr.find(f'{{{NS_A}}}solidFill')
#                                     if old_fill is not None: defRPr.remove(old_fill)
#                                     nf = etree.SubElement(defRPr, f'{{{NS_A}}}solidFill')
#                                     etree.SubElement(nf, f'{{{NS_A}}}srgbClr').set('val', HIGHLIGHT_HEX)
#                         print(f"      Restyled label at idx={point_idx}")
#                         break
#             for ex in ser_el.findall(f'{{{NS_C}}}dPt'):
#                 ie = ex.find(f'{{{NS_C}}}idx')
#                 if ie is not None and ie.get('val') == str(point_idx): ser_el.remove(ex); break
#             dPt = etree.Element(f'{{{NS_C}}}dPt')
#             etree.SubElement(dPt, f'{{{NS_C}}}idx').set('val', str(point_idx))
#             marker = etree.SubElement(dPt, f'{{{NS_C}}}marker')
#             etree.SubElement(marker, f'{{{NS_C}}}symbol').set('val', 'circle')
#             etree.SubElement(marker, f'{{{NS_C}}}size').set('val', '12')
#             spPr = etree.SubElement(marker, f'{{{NS_C}}}spPr')
#             sf = etree.SubElement(spPr, f'{{{NS_A}}}solidFill')
#             etree.SubElement(sf, f'{{{NS_A}}}srgbClr').set('val', 'FFFFFF')
#             ln = etree.SubElement(spPr, f'{{{NS_A}}}ln'); ln.set('w', '25400')
#             lf = etree.SubElement(ln, f'{{{NS_A}}}solidFill')
#             etree.SubElement(lf, f'{{{NS_A}}}srgbClr').set('val', '000000')
#             etree.SubElement(ln, f'{{{NS_A}}}prstDash').set('val', 'dash')
#             ib = None
#             for tag in ['dLbls','trendline','cat','val','xVal','yVal','smooth','extLst']:
#                 el = ser_el.find(f'{{{NS_C}}}{tag}')
#                 if el is not None: ib = el; break
#             if ib is not None: ser_el.insert(list(ser_el).index(ib), dPt)
#             else: ser_el.append(dPt)
#             print(f"      Added marker at idx={point_idx}")
#             break


# def hl_apply_line_callouts(chart, chart_shape, slide, highlight_spec, data_df=None):
#     highlights = highlight_spec.get('highlights', [])
#     box_w, box_h, gap = Cm(2.5), Cm(0.45), Cm(0.15)

#     callout_info = []
#     for hl in highlights:
#         if not hl.get('callout_text'): continue
#         cat_idx = hl_find_category_index_from_df(data_df, hl.get('category', ''))
#         if cat_idx is None: continue
#         pos = hl_calc_line_point_rect(chart, chart_shape, data_df, cat_idx, hl.get('series', ''))
#         if pos is None: continue
#         callout_info.append({
#             'cx': pos['center_x'],
#             'cy': pos['center_y'],
#             'text': hl['callout_text'],
#         })

#     callout_info.sort(key=lambda c: c['cy'])

#     placed = []
#     for info in callout_info:
#         cx, cy = info['cx'], info['cy']
#         box_left = cx - box_w // 2
#         box_top = cy + Cm(0.25)
#         for (pt, pb) in placed:
#             if box_top < pb + Cm(0.05):
#                 box_top = pb + gap
#         placed.append((box_top, box_top + box_h))

#         txBox = slide.shapes.add_textbox(
#             left=int(box_left), top=int(box_top),
#             width=int(box_w), height=int(box_h))
#         txBox.fill.solid()
#         txBox.fill.fore_color.rgb = RGBColor(0, 0, 0)
#         txBox.line.fill.background()
#         tf = txBox.text_frame
#         tf.word_wrap = True
#         tf.margin_top = tf.margin_bottom = Cm(0.03)
#         tf.margin_left = tf.margin_right = Cm(0.1)
#         p = tf.paragraphs[0]
#         p.alignment = PP_ALIGN.CENTER
#         run = p.add_run()
#         run.text = info['text']
#         run.font.size = Pt(7)
#         run.font.bold = True
#         run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
#         txBox.name = "HL_Callout"
#         print(f"      Callout below marker: {info['text']}")


# def hl_highlight_bar_segment(chart, chart_shape, slide, highlight_spec, data_df=None, direction='vertical'):
#     for hl in highlight_spec.get('highlights', []):
#         cat_idx = hl_find_category_index_from_df(data_df, hl.get('category', ''))
#         if cat_idx is None:
#             print(f"    Category '{hl.get('category')}' not found"); continue
#         print(f"    Bar highlight: '{hl.get('series')}' cat_idx={cat_idx}")
#         rect = hl_calc_segment_rect(chart, chart_shape, data_df, cat_idx, hl.get('series', ''), direction)
#         if rect:
#             hl_add_rectangle_overlay(slide, rect)
#             if hl.get('callout_text'):
#                 hl_add_callout_textbox(slide, rect, hl['callout_text'], hl.get('callout_position', 'right'), direction)
#         else:
#             print(f"      Could not calculate position")


# def remove_existing_highlights(prs):
#     removed = 0
#     for slide in prs.slides:
#         to_remove = [s for s in slide.shapes if s.name and s.name.startswith(("HL_Rect","HL_Callout","HL_Border","HL_Circle","HL_Leader"))]
#         for s in to_remove:
#             s._element.getparent().remove(s._element); removed += 1
#     if removed: print(f"  Removed {removed} old highlight shapes")


# def apply_all_highlights(prs, highlight_df, chart_data_map=None):
#     if highlight_df is None or highlight_df.empty:
#         print("No highlights to apply"); return
#     remove_existing_highlights(prs)
#     hl_rows = highlight_df[highlight_df['highlight_spec'].notna()]
#     if hl_rows.empty: return
#     has_csn = 'chart_shape_name' in hl_rows.columns
#     print(f"\n{'='*60}\nAPPLYING HIGHLIGHTS — {len(hl_rows)} charts\n{'='*60}")

#     for _, row in hl_rows.iterrows():
#         slide_num = int(row['slide_number'])
#         shape_name = str(row['chart_shape_name']).strip() if has_csn and pd.notna(row.get('chart_shape_name')) and str(row['chart_shape_name']).strip() else row['shape_name']
#         spec = row['highlight_spec']
#         if isinstance(spec, str):
#             try: spec = json.loads(spec)
#             except: continue
#         if not spec or 'highlights' not in spec: continue

#         slide_idx = slide_num - 1
#         if slide_idx < 0 or slide_idx >= len(prs.slides): continue
#         chart_shape = None
#         for s in prs.slides[slide_idx].shapes:
#             if s.name == shape_name: chart_shape = s; break
#         if not chart_shape or not chart_shape.has_chart: continue

#         detected = hl_detect_chart_type(chart_shape.chart)
#         data_df = chart_data_map.get((slide_num, shape_name)) if chart_data_map else None
#         print(f"\n  '{shape_name}' slide {slide_num} (detected: {detected})")

#         slide = prs.slides[slide_idx]
#         if detected == 'line':
#             hl_restyle_line_labels(chart_shape.chart, spec, data_df)
#             hl_apply_line_callouts(chart_shape.chart, chart_shape, slide, spec, data_df)
#         elif detected == 'horizontal_bar':
#             hl_highlight_bar_segment(chart_shape.chart, chart_shape, slide, spec, data_df, 'horizontal')
#         elif detected in ('stacked_bar', 'bar'):
#             hl_highlight_bar_segment(chart_shape.chart, chart_shape, slide, spec, data_df, 'vertical')
#         else:
#             print(f"    Unsupported: {detected}")

#     print(f"\nHighlights applied")


# # ── Read highlights from DB (using spark.sql) ──
# def read_highlights_from_db():
#     query = f"""SELECT slide_number, chart_id, shape_name, chart_shape_name, chart_type, highlight_spec
#                 FROM {COMBINED_TABLE} WHERE highlight_spec IS NOT NULL AND type = 'insight'"""
#     try:
#         df = spark.sql(query).toPandas()
#         print(f"  Read {len(df)} highlight rows from DB")
#         return df
#     except Exception as e:
#         print(f"  ERROR reading highlights: {e}")
#         return pd.DataFrame()


# # ============================================================
# # NEW ENTRY POINT: DECK UPDATE + HIGHLIGHTS FROM DB
# # ============================================================
# def run_deck_update_with_highlights(config_path, template_path, output_path,
#                                     max_parallel_queries=15,
#                                     update_embedded_excel=True):
#     chart_data_map = run_deck_update(
#         config_path, template_path, output_path,
#         max_parallel_queries, update_embedded_excel
#     )

#     print(f"\n{'='*60}\nREADING HIGHLIGHTS FROM DB\n{'='*60}")
#     highlight_df = read_highlights_from_db()

#     if highlight_df.empty:
#         print("No highlights found in DB — skipping")
#         return

#     prs = Presentation(output_path)
#     print(f"\nChart data map: {len(chart_data_map)} charts")
#     apply_all_highlights(prs, highlight_df, chart_data_map)
#     prs.save(output_path)
#     print(f"\nFinal deck saved: {output_path}")


# # ============================================================
# # USAGE
# # ============================================================
# #
# # Option A — Deck update only:
# #   run_deck_update(config, template, output)
# #
# # Option B — Deck update + highlights from DB:
# #   run_deck_update_with_highlights(config, template, output)

In [0]:
"""
Deck Update Pipeline + Highlight Application (Volumes version)
--------------------------------------------------------------
Converted from S3/SQL-connector notebook to Volumes-based output.
- Uses spark.sql() instead of databricks SQL connector
- Synchronous execution (no async/await)
- Outputs to Databricks Volumes
- Returns path via dbutils.notebook.exit()

Two entry points:
  run_deck_update()                    — Data updates only
  run_deck_update_with_highlights()    — Data updates + highlights + speaker notes
"""

import pandas as pd
from pptx import Presentation
from pptx.util import Pt, Cm
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.enum.shapes import MSO_SHAPE
from lxml import etree
import re
import json
import math
import os
import shutil
import zipfile
import openpyxl
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from decimal import Decimal


# ============================================================
# XML NAMESPACES
# ============================================================
CHART_NS = {
    'c': 'http://schemas.openxmlformats.org/drawingml/2006/chart',
    'a': 'http://schemas.openxmlformats.org/drawingml/2006/main',
    'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
}
NS_C = 'http://schemas.openxmlformats.org/drawingml/2006/chart'
NS_A = 'http://schemas.openxmlformats.org/drawingml/2006/main'


# ============================================================
# 1. CONFIG READER
# ============================================================
def read_config(config_path: str) -> pd.DataFrame:
    df = pd.read_excel(config_path)
    print(f"Config loaded: {len(df)} elements to update")
    print(df[['slide_num', 'shape_name', 'element_type']].to_string())
    return df


# ============================================================
# 2. DATA FETCHER  (spark.sql — no token needed)
# ============================================================
def fetch_data(query_name, sql_query):
    """Execute a SQL query via spark.sql and return a pandas DataFrame."""
    print(f"  -> Fetching: {query_name}")
    try:
        df = spark.sql(sql_query).toPandas()
        if 'dsg_deck' in df.columns and 'json_format_df' in df.columns:
            if len(df) > 0:
                df = pd.read_json(df.iloc[0]['json_format_df'], orient='records')
        print(f"     {len(df)} rows for {query_name}")
        return df
    except Exception as e:
        print(f"     ERROR fetching {query_name}: {e}")
        raise


def fetch_all_data(config_df, max_parallel=15):
    """Fetch all SQL data in parallel using ThreadPoolExecutor."""
    data_cache = {}
    sql_rows = config_df[config_df['sql_query'].notna() & (config_df['sql_query'].str.strip() != '')]
    unique_sqls = sql_rows['sql_query'].drop_duplicates().tolist()
    if not unique_sqls:
        return data_cache

    print(f"\nExecuting {len(unique_sqls)} SQL queries (max {max_parallel} concurrent)...")

    sql_to_df = {}
    with ThreadPoolExecutor(max_workers=max_parallel, thread_name_prefix='sql_fetch') as pool:
        futures = {
            pool.submit(fetch_data, f"SQL_{i+1}", sq): sq
            for i, sq in enumerate(unique_sqls)
        }
        for future in as_completed(futures):
            sq = futures[future]
            try:
                sql_to_df[sq] = future.result()
            except Exception as e:
                print(f"  Query failed: {e}")

    for idx, row in config_df.iterrows():
        sq = row.get('sql_query', '')
        data_cache[idx] = sql_to_df.get(sq, pd.DataFrame()) if pd.notna(sq) and str(sq).strip() else pd.DataFrame()

    print(f"Data cache built: {len(data_cache)} datasets")
    return data_cache


# ============================================================
# 3. SHAPE FINDER
# ============================================================
def find_shape(prs, slide_num, shape_name):
    slide_index = slide_num - 1
    if slide_index < 0 or slide_index >= len(prs.slides):
        return None
    for shape in prs.slides[slide_index].shapes:
        if shape.name == shape_name:
            return shape
    print(f"  Shape '{shape_name}' not found on slide {slide_num}")
    return None


# ============================================================
# 4. DATA TYPE CONVERTERS
# ============================================================
def date_string_to_excel_serial(date_str):
    for fmt in ['%b-%y', '%B-%y', '%b-%Y', '%Y-%m-%d', '%m/%d/%Y']:
        try:
            dt = datetime.strptime(date_str, fmt)
            return float((dt - datetime(1899, 12, 30)).days)
        except ValueError:
            continue
    try:
        return float(date_str)
    except (ValueError, TypeError):
        return date_str

def is_null_value(val):
    if val is None: return True
    if isinstance(val, float) and math.isnan(val): return True
    if isinstance(val, str) and val.strip() == '': return True
    return False

def convert_to_float(val):
    if is_null_value(val): return None
    if isinstance(val, Decimal): return float(val)
    if isinstance(val, (int, float)): return float(val)
    try: return float(val)
    except: return None

def detect_category_type(chart):
    tree = chart._chartSpace
    ser_elements = tree.findall('.//c:ser', CHART_NS)
    if ser_elements:
        cat_el = ser_elements[0].find('c:cat', CHART_NS)
        if cat_el is not None:
            if cat_el.find('c:numRef', CHART_NS) is not None: return 'numeric'
            if cat_el.find('c:strRef', CHART_NS) is not None: return 'string'
    return 'string'

def prepare_chart_data(chart, df):
    cat_type = detect_category_type(chart)
    raw_cats = df.iloc[:, 0].tolist()
    if cat_type == 'numeric':
        categories = []
        for v in raw_cats:
            if is_null_value(v): categories.append(None)
            elif isinstance(v, str): categories.append(date_string_to_excel_serial(v))
            elif isinstance(v, (int, float, Decimal)): categories.append(float(v))
            else: categories.append(v)
    else:
        categories = [str(v) if not is_null_value(v) else None for v in raw_cats]
    series_data = [[convert_to_float(v) for v in df.iloc[:, c].tolist()] for c in range(1, len(df.columns))]
    return categories, series_data


# ============================================================
# 5. CHART UPDATER
# ============================================================
def update_chart_data(chart, df):
    tree = chart._chartSpace
    categories, series_data = prepare_chart_data(chart, df)
    for cat_el in tree.findall('.//c:cat', CHART_NS):
        _update_ref_values(cat_el, categories)
    for i, ser_el in enumerate(tree.findall('.//c:ser', CHART_NS)):
        if i < len(series_data):
            val_el = ser_el.find('c:val', CHART_NS)
            if val_el is not None:
                _update_ref_values(val_el, series_data[i])
    print(f"  Chart updated: {len(categories)} cats x {len(series_data)} series")

def _update_ref_values(parent_el, values):
    for ref_type in ['c:numRef', 'c:strRef']:
        ref_el = parent_el.find(ref_type, CHART_NS)
        if ref_el is not None:
            f_el = ref_el.find('c:f', CHART_NS)
            if f_el is not None and f_el.text:
                f_el.text = _adjust_formula_range(f_el.text, len(values))
            cache_tag = 'c:numCache' if 'num' in ref_type else 'c:strCache'
            cache_el = ref_el.find(cache_tag, CHART_NS)
            if cache_el is not None:
                _replace_cache_values(cache_el, values)
                return
    for lit_type in ['c:numLit', 'c:strLit']:
        lit_el = parent_el.find(lit_type, CHART_NS)
        if lit_el is not None:
            _replace_cache_values(lit_el, values)
            return

def _adjust_formula_range(formula, data_len):
    m = re.match(r"(.+\$[A-Z]+\$)(\d+)(:\$[A-Z]+\$)(\d+)$", formula)
    if m:
        return f"{m.group(1)}{m.group(2)}{m.group(3)}{int(m.group(2)) + data_len - 1}"
    m2 = re.match(r"(.+\$)(\d+)$", formula)
    if m2:
        sm = re.search(r'\$(\d+):', formula)
        sr = int(sm.group(1)) if sm else 2
        return f"{m2.group(1)}{sr + data_len - 1}"
    return formula

def _replace_cache_values(cache_el, values):
    ns_c = CHART_NS['c']
    pt_count = cache_el.find('c:ptCount', CHART_NS)
    if pt_count is not None:
        pt_count.set('val', str(len(values)))
    for pt in cache_el.findall('c:pt', CHART_NS):
        cache_el.remove(pt)
    for idx, val in enumerate(values):
        if val is None: continue
        pt = etree.SubElement(cache_el, f'{{{ns_c}}}pt')
        pt.set('idx', str(idx))
        v = etree.SubElement(pt, f'{{{ns_c}}}v')
        v.text = repr(val) if isinstance(val, float) else str(val)


# ============================================================
# 5b. LAST-POINT DATA LABELS
# ============================================================
def add_last_point_data_labels(chart, df):
    for plot in chart.plots:
        for series in plot.series:
            ser_el = series._element
            is_dashed = False
            spPr = ser_el.find(f'{{{NS_C}}}spPr')
            if spPr is not None:
                ln = spPr.find(f'{{{NS_A}}}ln')
                if ln is not None:
                    pd_el = ln.find(f'{{{NS_A}}}prstDash')
                    if pd_el is not None and pd_el.get('val', '') not in ('', 'solid'): is_dashed = True
                    if ln.find(f'{{{NS_A}}}custDash') is not None: is_dashed = True
            if is_dashed: continue
            hex_color = None
            if spPr is not None:
                ln = spPr.find(f'{{{NS_A}}}ln')
                if ln is not None:
                    sf = ln.find(f'{{{NS_A}}}solidFill')
                    if sf is not None:
                        sc = sf.find(f'{{{NS_A}}}srgbClr')
                        if sc is not None: hex_color = sc.get('val')
            last_idx = 0
            for vt in ['val', 'yVal']:
                ve = ser_el.find(f'{{{NS_C}}}{vt}')
                if ve is not None:
                    for cp in [f'{{{NS_C}}}numRef/{{{NS_C}}}numCache', f'{{{NS_C}}}numLit']:
                        cache = ve.find(cp)
                        if cache is not None:
                            pts = cache.findall(f'{{{NS_C}}}pt')
                            if pts: last_idx = max(int(p.get('idx', '0')) for p in pts)
                    break
            old = ser_el.find(f'{{{NS_C}}}dLbls')
            if old is not None: ser_el.remove(old)
            dLbls = etree.Element(f'{{{NS_C}}}dLbls')
            ib = None
            for tag in ['cat', 'val', 'xVal', 'yVal', 'bubbleSize', 'smooth', 'extLst']:
                el = ser_el.find(f'{{{NS_C}}}{tag}')
                if el is not None: ib = el; break
            if ib is not None: ser_el.insert(list(ser_el).index(ib), dLbls)
            else: ser_el.append(dLbls)
            dLbl = etree.SubElement(dLbls, f'{{{NS_C}}}dLbl')
            etree.SubElement(dLbl, f'{{{NS_C}}}idx').set('val', str(last_idx))
            nf = etree.SubElement(dLbl, f'{{{NS_C}}}numFmt')
            nf.set('formatCode', '0%'); nf.set('sourceLinked', '0')
            etree.SubElement(dLbl, f'{{{NS_C}}}dLblPos').set('val', 'r')
            txPr = etree.SubElement(dLbl, f'{{{NS_C}}}txPr')
            etree.SubElement(txPr, f'{{{NS_A}}}bodyPr')
            etree.SubElement(txPr, f'{{{NS_A}}}lstStyle')
            p = etree.SubElement(txPr, f'{{{NS_A}}}p')
            pPr = etree.SubElement(p, f'{{{NS_A}}}pPr')
            rPr = etree.SubElement(pPr, f'{{{NS_A}}}defRPr')
            rPr.set('b', '1'); rPr.set('sz', '900'); rPr.set('lang', 'en-US')
            if hex_color:
                sf2 = etree.SubElement(rPr, f'{{{NS_A}}}solidFill')
                etree.SubElement(sf2, f'{{{NS_A}}}srgbClr').set('val', hex_color)
            etree.SubElement(p, f'{{{NS_A}}}endParaRPr').set('lang', 'en-US')
            for tag, val in [('showLegendKey','0'),('showVal','1'),('showCatName','0'),
                             ('showSerName','0'),('showPercent','0'),('showBubbleSize','0')]:
                etree.SubElement(dLbl, f'{{{NS_C}}}{tag}').set('val', val)
            nf2 = etree.SubElement(dLbls, f'{{{NS_C}}}numFmt')
            nf2.set('formatCode', '0%'); nf2.set('sourceLinked', '0')
            for tag in ['showLegendKey','showVal','showCatName','showSerName','showPercent','showBubbleSize']:
                etree.SubElement(dLbls, f'{{{NS_C}}}{tag}').set('val', '0')
            print(f"    Label idx={last_idx}, color=#{hex_color}")
    print(f"  Labels applied")


# ============================================================
# 6. TEXTBOX UPDATER
# ============================================================
def update_textbox(shape, df, static_text=None):
    if static_text and str(static_text).strip() and not pd.isna(static_text):
        display_text = str(static_text)
    elif not df.empty:
        val = df.iloc[0, 0]
        if isinstance(val, Decimal): val = float(val)
        display_text = str(val)
    else:
        return False
    if not shape.has_text_frame: return False
    tf = shape.text_frame
    for para in tf.paragraphs:
        if para.runs:
            para.runs[0].text = display_text
            for run in para.runs[1:]: run.text = ''
            # Triangle color coding
            if '▼' in display_text or '▾' in display_text:
                para.runs[0].font.color.rgb = RGBColor(0xFF, 0x00, 0x00)
            elif '▲' in display_text or '▴' in display_text:
                para.runs[0].font.color.rgb = RGBColor(0x00, 0x80, 0x00)
            break
    print(f"  Textbox updated: '{display_text}'")
    return True


# ============================================================
# 6b. EXEC SUMMARY TEXTBOX UPDATER
# ============================================================
def update_exec_summary_textbox(shape, text, title_font_size=9, bullet_font_size=8):
    if not shape.has_text_frame:
        print(f"  Shape has no text frame")
        return False

    tf = shape.text_frame
    NS_A_URI = 'http://schemas.openxmlformats.org/drawingml/2006/main'

    template_font_name = 'Calibri'
    template_font_color = None
    if tf.paragraphs and tf.paragraphs[0].runs:
        tmpl_run = tf.paragraphs[0].runs[0]
        if tmpl_run.font.name:
            template_font_name = tmpl_run.font.name
        try:
            if tmpl_run.font.color and tmpl_run.font.color.type is not None:
                template_font_color = str(tmpl_run.font.color.rgb)
        except (AttributeError, TypeError):
            template_font_color = None

    p_elements = list(tf._txBody.findall(f'{{{NS_A_URI}}}p'))
    for p_el in p_elements:
        tf._txBody.remove(p_el)

    lines = text.split('\n')
    content_lines = [l for l in lines if l.strip()]

    if not content_lines:
        print(f"  No content lines to write")
        return False

    for i, line in enumerate(content_lines):
        stripped = line.strip()
        is_bullet = stripped.startswith('- ')

        p_el = etree.SubElement(tf._txBody, f'{{{NS_A_URI}}}p')
        pPr = etree.SubElement(p_el, f'{{{NS_A_URI}}}pPr')

        if is_bullet:
            pPr.set('marL', '180000')
            pPr.set('indent', '-90000')
            spc_before = etree.SubElement(pPr, f'{{{NS_A_URI}}}spcBef')
            spc_pts = etree.SubElement(spc_before, f'{{{NS_A_URI}}}spcPts')
            spc_pts.set('val', '100')

            r = etree.SubElement(p_el, f'{{{NS_A_URI}}}r')
            rPr = etree.SubElement(r, f'{{{NS_A_URI}}}rPr')
            rPr.set('lang', 'en-US')
            rPr.set('sz', str(bullet_font_size * 100))
            rPr.set('b', '0')
            rPr.set('dirty', '0')

            latin = etree.SubElement(rPr, f'{{{NS_A_URI}}}latin')
            latin.set('typeface', template_font_name)

            if template_font_color:
                solidFill = etree.SubElement(rPr, f'{{{NS_A_URI}}}solidFill')
                srgbClr = etree.SubElement(solidFill, f'{{{NS_A_URI}}}srgbClr')
                srgbClr.set('val', template_font_color)

            t = etree.SubElement(r, f'{{{NS_A_URI}}}t')
            t.text = stripped
        else:
            if i > 0:
                spc_before = etree.SubElement(pPr, f'{{{NS_A_URI}}}spcBef')
                spc_pts = etree.SubElement(spc_before, f'{{{NS_A_URI}}}spcPts')
                spc_pts.set('val', '600')

            r = etree.SubElement(p_el, f'{{{NS_A_URI}}}r')
            rPr = etree.SubElement(r, f'{{{NS_A_URI}}}rPr')
            rPr.set('lang', 'en-US')
            rPr.set('sz', str(title_font_size * 100))
            rPr.set('b', '1')
            rPr.set('dirty', '0')

            latin = etree.SubElement(rPr, f'{{{NS_A_URI}}}latin')
            latin.set('typeface', template_font_name)

            if template_font_color:
                solidFill = etree.SubElement(rPr, f'{{{NS_A_URI}}}solidFill')
                srgbClr = etree.SubElement(solidFill, f'{{{NS_A_URI}}}srgbClr')
                srgbClr.set('val', template_font_color)

            t = etree.SubElement(r, f'{{{NS_A_URI}}}t')
            t.text = stripped

    title_count = len([l for l in content_lines if not l.strip().startswith('- ')])
    bullet_count = len([l for l in content_lines if l.strip().startswith('- ')])
    print(f"  Exec summary updated: {len(content_lines)} lines "
          f"({title_count} titles bold, {bullet_count} bullets indented)")
    return True


# ============================================================
# 7. TABLE / EXCEL UPDATER
# ============================================================
def update_table(shape, df):
    if not shape.has_table: return False
    table = shape.table
    for r_idx, row_data in enumerate(df.values):
        if r_idx >= len(table.rows): break
        for c_idx, val in enumerate(row_data):
            if c_idx >= len(table.columns): break
            cell = table.cell(r_idx, c_idx)
            if is_null_value(val): val = ''
            elif isinstance(val, Decimal): val = float(val)
            if cell.text_frame.paragraphs and cell.text_frame.paragraphs[0].runs:
                cell.text_frame.paragraphs[0].runs[0].text = str(val)
                for run in cell.text_frame.paragraphs[0].runs[1:]: run.text = ''
            else:
                cell.text = str(val)
    print(f"  Table updated: {len(df)}r x {len(df.columns)}c")
    return True

def get_chart_excel_mapping(pptx_path):
    mapping = {}
    with zipfile.ZipFile(pptx_path, 'r') as zf:
        for name in zf.namelist():
            if name.startswith('ppt/charts/_rels/') and name.endswith('.rels'):
                chart_name = name.split('/')[-1].replace('.rels', '')
                rels_xml = zf.read(name)
                for rel in etree.fromstring(rels_xml).findall(
                    './/{http://schemas.openxmlformats.org/package/2006/relationships}Relationship'):
                    target = rel.get('Target')
                    if target and 'embeddings' in target:
                        mapping[chart_name] = target.split('/')[-1]
    return mapping

def update_embedded_excel_in_pptx(pptx_path, output_path, chart_excel_data):
    temp_dir = 'temp_pptx_extract'
    if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
    with zipfile.ZipFile(pptx_path, 'r') as zf: zf.extractall(temp_dir)
    embeddings_dir = os.path.join(temp_dir, 'ppt', 'embeddings')
    if os.path.exists(embeddings_dir):
        for excel_name, df in chart_excel_data.items():
            excel_path = os.path.join(embeddings_dir, excel_name)
            if not os.path.exists(excel_path) or df.empty: continue
            try:
                wb = openpyxl.load_workbook(excel_path)
                ws = wb.active
                merged = set()
                for mr in ws.merged_cells.ranges:
                    for r in range(mr.min_row, mr.max_row+1):
                        for c in range(mr.min_col, mr.max_col+1):
                            if not (r == mr.min_row and c == mr.min_col): merged.add((r, c))
                for r in range(2, ws.max_row+1):
                    for c in range(1, ws.max_column+1):
                        if (r, c) not in merged:
                            try: ws.cell(row=r, column=c, value=None)
                            except: pass
                for r_idx, row_data in enumerate(df.values):
                    for c_idx, val in enumerate(row_data):
                        tr, tc = 2+r_idx, 1+c_idx
                        if (tr, tc) in merged: continue
                        if is_null_value(val): val = None
                        elif isinstance(val, Decimal): val = float(val)
                        elif isinstance(val, str):
                            for fmt in ['%b-%y','%B-%y','%Y-%m-%d','%b-%Y']:
                                try: val = datetime.strptime(val, fmt); break
                                except: continue
                        try: ws.cell(row=tr, column=tc, value=val)
                        except: continue
                wb.save(excel_path)
                print(f"    Updated: {excel_name}")
            except Exception as e:
                print(f"    Failed: {excel_name} - {e}")
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(temp_dir):
            for f in files:
                fp = os.path.join(root, f)
                zf.write(fp, os.path.relpath(fp, temp_dir))
    shutil.rmtree(temp_dir)


# ============================================================
# 8. ELEMENT UPDATER
# ============================================================
def update_element(prs, slide_num, shape_name, element_type, df, static_text=None, show_last_label=False):
    shape = find_shape(prs, slide_num, shape_name)
    if shape is None: return False, None
    if element_type == 'chart':
        if df.empty or not shape.has_chart: return False, None
        chart = shape.chart
        chart_xml_name = chart.part.partname.split('/')[-1]
        update_chart_data(chart, df)
        if show_last_label: add_last_point_data_labels(chart, df)
        return True, chart_xml_name
    elif element_type == 'textbox':
        return update_textbox(shape, df, static_text), None
    elif element_type == 'exec_summary':
        display_text = None
        if static_text and str(static_text).strip() and not pd.isna(static_text):
            display_text = str(static_text)
        elif not df.empty:
            val = df.iloc[0, 0]
            if isinstance(val, Decimal): val = float(val)
            display_text = str(val)
        if not display_text or display_text.strip() == '' or display_text == 'nan':
            print(f"  No exec summary text to update")
            return False, None
        ok = update_exec_summary_textbox(shape, display_text)
        return ok, None
    elif element_type == 'table':
        if df.empty or not shape.has_table: return False, None
        return update_table(shape, df), None
    return False, None


# ============================================================
# 9. MAIN ORCHESTRATOR (synchronous, Volumes output)
# ============================================================
def run_deck_update(config_path, template_path, output_path,
                    max_parallel_queries=15, update_embedded_excel=True):
    print("=" * 60 + "\nDECK UPDATE STARTED\n" + "=" * 60)
    config_df = read_config(config_path)
    data_cache = fetch_all_data(config_df, max_parallel_queries)
    prs = Presentation(template_path)
    print(f"\nTemplate: {len(prs.slides)} slides")
    chart_excel_mapping = get_chart_excel_mapping(template_path) if update_embedded_excel else {}
    success, fail, chart_data_for_excel, chart_data_map = 0, 0, {}, {}

    for idx, row in config_df.iterrows():
        slide_num = int(row['slide_num'])
        try:
            static_text = row.get('static_text', None)
            if pd.isna(static_text): static_text = None
            slr = row.get('show_last_label', False)
            show_last_label = False if pd.isna(slr) else str(slr).strip().upper() in ('TRUE','1','1.0','YES')
            df = data_cache.get(idx, pd.DataFrame())
            ok, chart_xml_name = debug_wrap_update(
                                    prs, slide_num, row['shape_name'], row['element_type'],
                                    df, static_text, show_last_label
                                )
            if ok:
                success += 1
                if row['element_type'] == 'chart' and not df.empty:
                    chart_data_map[(slide_num, row['shape_name'])] = df
                if update_embedded_excel and chart_xml_name and chart_xml_name in chart_excel_mapping:
                    chart_data_for_excel[chart_excel_mapping[chart_xml_name]] = df
            else:
                fail += 1
        except Exception as e:
            print(f"  Error: {e}")
            traceback.print_exc()
            fail += 1
            # ── Add slide insights to speaker notes ──
    try:
        notes_df = spark.sql(f"""
            SELECT DISTINCT slide_number, slide_insights
            FROM {COMBINED_TABLE}
            WHERE type IN ('heading', 'enhanced_heading')
              AND slide_insights IS NOT NULL
        """).toPandas()

        if not notes_df.empty:
            print(f"\nAdding slide insights to speaker notes ({len(notes_df)} slides)...")
            for _, nr in notes_df.iterrows():
                si = int(nr['slide_number']) - 1
                if si < 0 or si >= len(prs.slides):
                    continue
                raw_insights = str(nr.get('slide_insights', ''))
                lines = [l.strip() for l in raw_insights.split('\n') if l.strip() and l.strip() not in ('None', 'nan')]
                if not lines:
                    continue
                #notes_text = "SLIDE INSIGHTS\n" + "\n".join(lines)
                # notes_text = "Narrative:\n" + "\n".join(lines)
                # slide = prs.slides[si]
                # notes_slide = slide.notes_slide
                # notes_tf = notes_slide.notes_text_frame
                # existing = notes_tf.text.strip() if notes_tf.text else ""
                # notes_tf.text = (existing + "\n\n" + notes_text) if existing else notes_text
                slide = prs.slides[si]
                notes_slide = slide.notes_slide
                notes_tf = notes_slide.notes_text_frame
                existing = notes_tf.text.strip() if notes_tf.text else ""
                
                NS_A_URI = 'http://schemas.openxmlformats.org/drawingml/2006/main'

                if not existing:
                    for p_el in list(notes_tf._txBody.findall(f'{{{NS_A_URI}}}p')):
                        notes_tf._txBody.remove(p_el)

                if existing:
                    blank_p = etree.SubElement(notes_tf._txBody, f'{{{NS_A_URI}}}p')

                title_p = etree.SubElement(notes_tf._txBody, f'{{{NS_A_URI}}}p')
                title_r = etree.SubElement(title_p, f'{{{NS_A_URI}}}r')
                title_rPr = etree.SubElement(title_r, f'{{{NS_A_URI}}}rPr')
                title_rPr.set('lang', 'en-US')
                title_rPr.set('b', '1')
                title_t = etree.SubElement(title_r, f'{{{NS_A_URI}}}t')
                title_t.text = "Narrative:"

                for line in lines:
                    line_p = etree.SubElement(notes_tf._txBody, f'{{{NS_A_URI}}}p')
                    # Add bullet formatting
                    line_pPr = etree.SubElement(line_p, f'{{{NS_A_URI}}}pPr')
                    line_pPr.set('marL', '171450')
                    line_pPr.set('indent', '-171450')
                    buChar = etree.SubElement(line_pPr, f'{{{NS_A_URI}}}buChar')
                    buChar.set('char', '\u2022')
                    line_r = etree.SubElement(line_p, f'{{{NS_A_URI}}}r')
                    line_rPr = etree.SubElement(line_r, f'{{{NS_A_URI}}}rPr')
                    line_rPr.set('lang', 'en-US')
                    line_rPr.set('b', '0')
                    line_t = etree.SubElement(line_r, f'{{{NS_A_URI}}}t')
                    line_t.text = line
                print(f"  Slide {int(nr['slide_number'])}: {len(lines)} insights → notes")
    except Exception as e:
        print(f"  Speaker notes error (non-fatal): {e}")

    # Save to LOCAL temp path first (Volumes FUSE doesn't support zipfile close)
    local_temp = "/tmp/_deck_temp.pptx"
    local_final = "/tmp/_deck_final.pptx"
    prs.save(local_temp)
    if update_embedded_excel and chart_data_for_excel:
        update_embedded_excel_in_pptx(local_temp, local_final, chart_data_for_excel)
        os.remove(local_temp)
    else:
        shutil.move(local_temp, local_final)

    # Copy bytes to Volumes
    with open(local_final, 'rb') as f:
        pptx_bytes = f.read()
    with open(output_path, 'wb') as f:
        f.write(pptx_bytes)
    os.remove(local_final)
    print(f"\nSuccess: {success}, Failed: {fail}, Saved to: {output_path}")
    return chart_data_map


# ################################################################
# ##  HIGHLIGHT APPLICATION — APPENDED BELOW (core untouched)   ##
# ################################################################

HIGHLIGHT_HEX = '000000'
COMBINED_TABLE = "`commercial-us-apo-onc-iia-dev`.`infigen_adhocs`.Narrative_deck_insights_enhanced"


def hl_detect_chart_type(chart):
    tree = chart._chartSpace
    pa = tree.find('.//c:plotArea', CHART_NS)
    if pa is None: return 'unknown'
    for child in pa:
        tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
        if tag in ('lineChart', 'line3DChart'): return 'line'
        if tag in ('barChart', 'bar3DChart'):
            bd = child.find('c:barDir', CHART_NS)
            return 'horizontal_bar' if bd is not None and bd.get('val') == 'bar' else 'stacked_bar'
        if tag in ('areaChart', 'area3DChart'): return 'area'
        if tag in ('pieChart', 'pie3DChart', 'doughnutChart'): return 'pie'
        if tag == 'scatterChart': return 'scatter'
    return 'unknown'


def hl_find_category_index_from_df(data_df, target_category):
    if data_df is None or data_df.empty: return None
    target = target_category.strip()
    for i, cat in enumerate(data_df.iloc[:, 0].tolist()):
        cs = str(cat).strip()
        if cs == target or cs.startswith(target) or target in cs or cs in target:
            return i
    return None

def hl_get_series_name(ser_el):
    tx = ser_el.find('c:tx', CHART_NS)
    if tx is not None:
        for v in tx.iter():
            if v.text and v.tag.endswith('}v'): return v.text.strip()
    return None

def hl_matches_series(a, b):
    if not a or not b: return False
    a, b = a.strip().upper(), b.strip().upper()
    return a == b or a in b or b in a

def hl_find_point_index_in_series(ser_el, cat_idx):
    for vt in ['c:val', 'c:yVal']:
        ve = ser_el.find(vt, CHART_NS)
        if ve is not None:
            for ct in ['c:numCache', 'c:numLit']:
                cache = ve.find(ct, CHART_NS)
                if cache is not None:
                    indices = sorted(int(p.get('idx','0')) for p in cache.findall('c:pt', CHART_NS))
                    if cat_idx in indices: return cat_idx
                    valid = [i for i in indices if i <= cat_idx]
                    return valid[-1] if valid else (indices[-1] if indices else cat_idx)
            break
    return cat_idx

def hl_parse_value(val):
    if val is None: return None
    if isinstance(val, Decimal): return float(val)
    if isinstance(val, (int, float)):
        return None if math.isnan(val) else float(val)
    if isinstance(val, str):
        val = val.strip()
        if val in ('', 'nan', 'none', 'None'): return None
        if '%' in val:
            try: return float(val.replace('%','').strip()) / 100.0
            except: return None
        try: return float(val)
        except: return None
    return None


def hl_read_plot_area(chart, chart_shape):
    tree = chart._chartSpace
    sl, st, sw, sh = chart_shape.left, chart_shape.top, chart_shape.width, chart_shape.height
    pa = tree.find('.//c:plotArea', CHART_NS)
    if pa is not None:
        layout = pa.find('c:layout', CHART_NS)
        if layout is not None:
            ml = layout.find('c:manualLayout', CHART_NS)
            if ml is not None:
                x, y, w, h = [ml.find(f'c:{t}', CHART_NS) for t in ['x','y','w','h']]
                if all(e is not None for e in [x, y, w, h]):
                    try:
                        xf, yf, wf, hf = float(x.get('val','0')), float(y.get('val','0')), float(w.get('val','1')), float(h.get('val','1'))
                        pl = sl + int(xf * sw)
                        pt = st + int(yf * sh)
                        pw, ph = int(wf * sw), int(hf * sh)
                        return {'left': pl, 'top': pt, 'right': pl+pw, 'bottom': pt+ph, 'width': pw, 'height': ph}
                    except: pass
    pl = sl + int(sw*0.08); pr = sl+sw - int(sw*0.03)
    pt2 = st + int(sh*0.05); pb = st+sh - int(sh*0.15)
    return {'left': pl, 'top': pt2, 'right': pr, 'bottom': pb, 'width': pr-pl, 'height': pb-pt2}

def hl_read_gap_width(chart):
    tree = chart._chartSpace
    for ct in ['c:barChart', 'c:bar3DChart']:
        ce = tree.find(f'.//{ct}', CHART_NS)
        if ce is not None:
            gw = ce.find('c:gapWidth', CHART_NS)
            if gw is not None:
                try: return int(gw.get('val', '150'))
                except: pass
    return 150


def hl_calc_segment_rect(chart, chart_shape, data_df, cat_idx, target_series, direction='vertical'):
    if data_df is None or data_df.empty: return None
    plot = hl_read_plot_area(chart, chart_shape)
    gap_pct = hl_read_gap_width(chart)
    n_cats = data_df.shape[0]
    series_names = list(data_df.columns[1:])

    tree_cb = chart._chartSpace
    cross_between = 'between'
    for at in ['c:catAx', 'c:dateAx']:
        for ax in tree_cb.findall(f'.//{at}', CHART_NS):
            cb = ax.find('c:crossBetween', CHART_NS)
            if cb is not None: cross_between = cb.get('val', 'between')

    if direction == 'horizontal':
        cat_span, val_span = plot['height'], plot['width']
        cat_orientation = None
        for at in ['c:catAx', 'c:dateAx']:
            for ax in tree_cb.findall(f'.//{at}', CHART_NS):
                sc = ax.find('c:scaling', CHART_NS)
                if sc is not None:
                    ori = sc.find('c:orientation', CHART_NS)
                    if ori is not None:
                        cat_orientation = ori.get('val', '')
                        print(f"    [Rect] Found orientation={cat_orientation} on {at}")
        if cat_orientation is None:
            cat_orientation = 'minMax'
            print(f"    [Rect] No catAx orientation found — defaulting to minMax (reversed)")
        if cat_orientation == 'minMax':
            pos_cat_idx = n_cats - 1 - cat_idx
        else:
            pos_cat_idx = cat_idx
        print(f"    [Rect] horizontal: cat_idx={cat_idx}→pos={pos_cat_idx} (orientation={cat_orientation})")
    else:
        cat_span, val_span = plot['width'], plot['height']
        pos_cat_idx = cat_idx

    if cross_between == 'midCat' and n_cats > 1:
        bp = cat_span / (n_cats - 1)
        bt = int(bp / (1 + gap_pct/100.0))
        cc = int(pos_cat_idx * bp)
    else:
        bp = cat_span / n_cats
        bt = int(bp / (1 + gap_pct/100.0))
        cc = int((pos_cat_idx + 0.5) * bp)
    cs = cc - bt // 2
    tol = int(bt * 0.04); cs -= tol; bt += tol * 2

    axis_max = 1.0
    for ax in tree_cb.findall('.//c:valAx', CHART_NS):
        sc = ax.find('c:scaling', CHART_NS)
        if sc is not None:
            mx = sc.find('c:max', CHART_NS)
            if mx is not None:
                try: axis_max = float(mx.get('val', '1'))
                except: pass

    cum, seg_val, found = 0.0, 0.0, False
    for col in series_names:
        v = hl_parse_value(data_df.iloc[cat_idx][col])
        if v is None: v = 0.0
        if hl_matches_series(col, target_series):
            seg_val = v; found = True; break
        cum += v
    if not found or seg_val == 0: return None

    sf = cum / axis_max
    ef = (cum + seg_val) / axis_max

    if direction == 'horizontal':
        return {'left': plot['left'] + int(sf * val_span), 'top': plot['top'] + cs,
                'width': int((ef - sf) * val_span), 'height': bt}
    else:
        st_y = plot['bottom'] - int(ef * val_span)
        sb_y = plot['bottom'] - int(sf * val_span)
        return {'left': plot['left'] + cs, 'top': min(st_y, sb_y),
                'width': bt, 'height': abs(sb_y - st_y)}


def hl_calc_line_point_rect(chart, chart_shape, data_df, cat_idx, target_series):
    if data_df is None or data_df.empty: return None
    plot = hl_read_plot_area(chart, chart_shape)
    n_cats = data_df.shape[0]
    series_names = list(data_df.columns[1:])

    x_step = plot['width'] / max(n_cats - 1, 1)
    point_x = plot['left'] + int(cat_idx * x_step)

    target_val = None
    for col in series_names:
        if hl_matches_series(col, target_series):
            target_val = hl_parse_value(data_df.iloc[cat_idx][col])
            break
    if target_val is None: return None

    tree = chart._chartSpace
    axis_min = None
    axis_max = None
    for ax in tree.findall('.//c:valAx', CHART_NS):
        sc = ax.find('c:scaling', CHART_NS)
        if sc is not None:
            mn = sc.find('c:min', CHART_NS)
            mx = sc.find('c:max', CHART_NS)
            if mn is not None:
                try: axis_min = float(mn.get('val', '0'))
                except: pass
            if mx is not None:
                try: axis_max = float(mx.get('val', '1'))
                except: pass

    if axis_max is None:
        all_vals = []
        for col in series_names:
            for v in data_df[col].tolist():
                pv = hl_parse_value(v)
                if pv is not None: all_vals.append(pv)
        if all_vals:
            data_max = max(all_vals)
            if data_max <= 1:
                axis_max = math.ceil(data_max * 20) / 20
            else:
                axis_max = math.ceil(data_max * 10) / 10
        else:
            axis_max = 1.0
    if axis_min is None:
        axis_min = 0.0

    value_range = axis_max - axis_min if (axis_max - axis_min) != 0 else 1
    point_y = plot['bottom'] - int(((target_val - axis_min) / value_range) * plot['height'])

    print(f"      [LinePos] series='{target_series}', val={target_val}, "
          f"axis=[{axis_min}, {axis_max}], y={point_y}")

    return {'left': point_x, 'top': point_y, 'width': 0, 'height': 0,
            'center_x': point_x, 'center_y': point_y}


def hl_add_rectangle_overlay(slide, rect, padding=Cm(0.1)):
    shape = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE,
        left=int(rect['left'] - padding), top=int(rect['top'] - padding),
        width=int(rect['width'] + padding*2), height=int(rect['height'] + padding*2))
    shape.fill.background()
    shape.line.color.rgb = RGBColor(0, 0, 0)
    shape.line.width = Pt(1.5)
    ln_el = shape._element.find(f'.//{{{NS_A}}}ln')
    if ln_el is not None:
        etree.SubElement(ln_el, f'{{{NS_A}}}prstDash').set('val', 'dash')
    shape.name = "HL_Rect"
    return shape

def hl_add_callout_textbox(slide, rect, callout_text, position='right', direction='vertical'):
    box_w, box_h, gap = Cm(2.5), Cm(0.45), Cm(0.1)
    if position == 'right':
        bl = rect['left'] + rect['width'] + gap
        bt = rect['top'] + rect['height']//2 - box_h//2
    elif position == 'left':
        bl = rect['left'] - box_w - gap
        bt = rect['top'] + rect['height']//2 - box_h//2
    elif position == 'top':
        bl = rect['left'] + rect['width']//2 - box_w//2
        bt = rect['top'] - box_h - gap
    elif position == 'bottom':
        bl = rect['left'] + rect['width']//2 - box_w//2
        bt = rect['top'] + rect['height'] + gap
    else:
        if direction == 'horizontal':
            bl = rect['left'] + rect['width']//2 - box_w//2
            bt = rect['top'] + rect['height'] + gap
        else:
            bl = rect['left'] + rect['width'] + gap
            bt = rect['top'] + rect['height']//2 - box_h//2
    txBox = slide.shapes.add_textbox(left=int(bl), top=int(bt), width=int(box_w), height=int(box_h))
    txBox.fill.solid()
    txBox.fill.fore_color.rgb = RGBColor(0, 0, 0)
    txBox.line.fill.background()
    tf = txBox.text_frame
    tf.word_wrap = True
    tf.margin_top = tf.margin_bottom = Cm(0.03)
    tf.margin_left = tf.margin_right = Cm(0.1)
    p = tf.paragraphs[0]
    p.alignment = PP_ALIGN.CENTER
    run = p.add_run()
    run.text = callout_text
    run.font.size = Pt(7)
    run.font.bold = True
    run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    txBox.name = "HL_Callout"
    return txBox


def hl_restyle_line_labels(chart, highlight_spec, data_df=None):
    highlights = highlight_spec.get('highlights', [])
    tree = chart._chartSpace
    for hl in highlights:
        target_series = hl.get('series', '')
        target_category = hl.get('category', '')
        cat_idx = hl_find_category_index_from_df(data_df, target_category)
        if cat_idx is None: continue
        for ser_el in tree.findall('.//c:ser', CHART_NS):
            sn = hl_get_series_name(ser_el)
            if not hl_matches_series(sn, target_series): continue
            point_idx = hl_find_point_index_in_series(ser_el, cat_idx)
            print(f"    Line highlight: '{sn}' idx={point_idx}")
            dLbls = ser_el.find(f'{{{NS_C}}}dLbls')
            if dLbls is not None:
                for dLbl in dLbls.findall(f'{{{NS_C}}}dLbl'):
                    idx_el = dLbl.find(f'{{{NS_C}}}idx')
                    if idx_el is not None and idx_el.get('val') == str(point_idx):
                        txPr = dLbl.find(f'{{{NS_C}}}txPr')
                        if txPr is not None:
                            for defRPr in txPr.iter():
                                if defRPr.tag == f'{{{NS_A}}}defRPr':
                                    defRPr.set('sz', '1000')
                                    old_fill = defRPr.find(f'{{{NS_A}}}solidFill')
                                    if old_fill is not None: defRPr.remove(old_fill)
                                    nf = etree.SubElement(defRPr, f'{{{NS_A}}}solidFill')
                                    etree.SubElement(nf, f'{{{NS_A}}}srgbClr').set('val', HIGHLIGHT_HEX)
                        print(f"      Restyled label at idx={point_idx}")
                        break
            for ex in ser_el.findall(f'{{{NS_C}}}dPt'):
                ie = ex.find(f'{{{NS_C}}}idx')
                if ie is not None and ie.get('val') == str(point_idx): ser_el.remove(ex); break
            dPt = etree.Element(f'{{{NS_C}}}dPt')
            etree.SubElement(dPt, f'{{{NS_C}}}idx').set('val', str(point_idx))
            marker = etree.SubElement(dPt, f'{{{NS_C}}}marker')
            etree.SubElement(marker, f'{{{NS_C}}}symbol').set('val', 'circle')
            etree.SubElement(marker, f'{{{NS_C}}}size').set('val', '12')
            spPr = etree.SubElement(marker, f'{{{NS_C}}}spPr')
            sf = etree.SubElement(spPr, f'{{{NS_A}}}solidFill')
            etree.SubElement(sf, f'{{{NS_A}}}srgbClr').set('val', 'FFFFFF')
            ln = etree.SubElement(spPr, f'{{{NS_A}}}ln'); ln.set('w', '25400')
            lf = etree.SubElement(ln, f'{{{NS_A}}}solidFill')
            etree.SubElement(lf, f'{{{NS_A}}}srgbClr').set('val', '000000')
            etree.SubElement(ln, f'{{{NS_A}}}prstDash').set('val', 'dash')
            ib = None
            for tag in ['dLbls','trendline','cat','val','xVal','yVal','smooth','extLst']:
                el = ser_el.find(f'{{{NS_C}}}{tag}')
                if el is not None: ib = el; break
            if ib is not None: ser_el.insert(list(ser_el).index(ib), dPt)
            else: ser_el.append(dPt)
            print(f"      Added marker at idx={point_idx}")
            break


def hl_apply_line_callouts(chart, chart_shape, slide, highlight_spec, data_df=None):
    highlights = highlight_spec.get('highlights', [])
    box_w, box_h, gap = Cm(2.5), Cm(0.45), Cm(0.15)

    callout_info = []
    for hl in highlights:
        if not hl.get('callout_text'): continue
        cat_idx = hl_find_category_index_from_df(data_df, hl.get('category', ''))
        if cat_idx is None: continue
        pos = hl_calc_line_point_rect(chart, chart_shape, data_df, cat_idx, hl.get('series', ''))
        if pos is None: continue
        callout_info.append({
            'cx': pos['center_x'],
            'cy': pos['center_y'],
            'text': hl['callout_text'],
        })

    callout_info.sort(key=lambda c: c['cy'])

    placed = []
    for info in callout_info:
        cx, cy = info['cx'], info['cy']
        box_left = cx - box_w // 2
        box_top = cy + Cm(0.25)
        for (pt, pb) in placed:
            if box_top < pb + Cm(0.05):
                box_top = pb + gap
        placed.append((box_top, box_top + box_h))

        txBox = slide.shapes.add_textbox(
            left=int(box_left), top=int(box_top),
            width=int(box_w), height=int(box_h))
        txBox.fill.solid()
        txBox.fill.fore_color.rgb = RGBColor(0, 0, 0)
        txBox.line.fill.background()
        tf = txBox.text_frame
        tf.word_wrap = True
        tf.margin_top = tf.margin_bottom = Cm(0.03)
        tf.margin_left = tf.margin_right = Cm(0.1)
        p = tf.paragraphs[0]
        p.alignment = PP_ALIGN.CENTER
        run = p.add_run()
        run.text = info['text']
        run.font.size = Pt(7)
        run.font.bold = True
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        txBox.name = "HL_Callout"
        print(f"      Callout below marker: {info['text']}")


def hl_highlight_bar_segment(chart, chart_shape, slide, highlight_spec, data_df=None, direction='vertical'):
    for hl in highlight_spec.get('highlights', []):
        cat_idx = hl_find_category_index_from_df(data_df, hl.get('category', ''))
        if cat_idx is None:
            print(f"    Category '{hl.get('category')}' not found"); continue
        print(f"    Bar highlight: '{hl.get('series')}' cat_idx={cat_idx}")
        rect = hl_calc_segment_rect(chart, chart_shape, data_df, cat_idx, hl.get('series', ''), direction)
        if rect:
            hl_add_rectangle_overlay(slide, rect)
            if hl.get('callout_text'):
                hl_add_callout_textbox(slide, rect, hl['callout_text'], hl.get('callout_position', 'right'), direction)
        else:
            print(f"      Could not calculate position")


def remove_existing_highlights(prs):
    removed = 0
    for slide in prs.slides:
        to_remove = [s for s in slide.shapes if s.name and s.name.startswith(("HL_Rect","HL_Callout","HL_Border","HL_Circle","HL_Leader"))]
        for s in to_remove:
            s._element.getparent().remove(s._element); removed += 1
    if removed: print(f"  Removed {removed} old highlight shapes")


def apply_all_highlights(prs, highlight_df, chart_data_map=None):
    if highlight_df is None or highlight_df.empty:
        print("No highlights to apply"); return
    remove_existing_highlights(prs)
    hl_rows = highlight_df[highlight_df['highlight_spec'].notna()]
    if hl_rows.empty: return
    has_csn = 'chart_shape_name' in hl_rows.columns
    print(f"\n{'='*60}\nAPPLYING HIGHLIGHTS — {len(hl_rows)} charts\n{'='*60}")

    for _, row in hl_rows.iterrows():
        slide_num = int(row['slide_number'])
        shape_name = str(row['chart_shape_name']).strip() if has_csn and pd.notna(row.get('chart_shape_name')) and str(row['chart_shape_name']).strip() else row['shape_name']
        spec = row['highlight_spec']
        if isinstance(spec, str):
            try: spec = json.loads(spec)
            except: continue
        if not spec or 'highlights' not in spec: continue

        slide_idx = slide_num - 1
        if slide_idx < 0 or slide_idx >= len(prs.slides): continue
        chart_shape = None
        for s in prs.slides[slide_idx].shapes:
            if s.name == shape_name: chart_shape = s; break
        if not chart_shape or not chart_shape.has_chart: continue

        detected = hl_detect_chart_type(chart_shape.chart)
        data_df = chart_data_map.get((slide_num, shape_name)) if chart_data_map else None
        print(f"\n  '{shape_name}' slide {slide_num} (detected: {detected})")

        slide = prs.slides[slide_idx]
        if detected == 'line':
            hl_restyle_line_labels(chart_shape.chart, spec, data_df)
            hl_apply_line_callouts(chart_shape.chart, chart_shape, slide, spec, data_df)
        elif detected == 'horizontal_bar':
            hl_highlight_bar_segment(chart_shape.chart, chart_shape, slide, spec, data_df, 'horizontal')
        elif detected in ('stacked_bar', 'bar'):
            hl_highlight_bar_segment(chart_shape.chart, chart_shape, slide, spec, data_df, 'vertical')
        else:
            print(f"    Unsupported: {detected}")

    print(f"\nHighlights applied")


# ── Read highlights from DB (using spark.sql) ──
def read_highlights_from_db():
    query = f"""SELECT slide_number, chart_id, shape_name, chart_shape_name, chart_type, highlight_spec
                FROM {COMBINED_TABLE} WHERE highlight_spec IS NOT NULL AND type = 'insight'"""
    try:
        df = spark.sql(query).toPandas()
        print(f"  Read {len(df)} highlight rows from DB")
        return df
    except Exception as e:
        print(f"  ERROR reading highlights: {e}")
        return pd.DataFrame()


# ============================================================
# NEW ENTRY POINT: DECK UPDATE + HIGHLIGHTS FROM DB
# ============================================================
def run_deck_update_with_highlights(config_path, template_path, output_path,
                                    max_parallel_queries=15,
                                    update_embedded_excel=True):
    chart_data_map = run_deck_update(
        config_path, template_path, output_path,
        max_parallel_queries, update_embedded_excel
    )

    print(f"\n{'='*60}\nREADING HIGHLIGHTS FROM DB\n{'='*60}")
    highlight_df = read_highlights_from_db()

    if highlight_df.empty:
        print("No highlights found in DB — skipping")
        return

    prs = Presentation(output_path)
    print(f"\nChart data map: {len(chart_data_map)} charts")
    apply_all_highlights(prs, highlight_df, chart_data_map)
    prs.save(output_path)
    print(f"\nFinal deck saved: {output_path}")


# ============================================================
# USAGE
# ============================================================
#
# Option A — Deck update only:
#   run_deck_update(config, template, output)
#
# Option B — Deck update + highlights from DB:
#   run_deck_update_with_highlights(config, template, output)

In [0]:
def debug_wrap_update(prs, slide_num, shape_name, element_type, df, static_text, show_last_label):
    print("\n----- DEBUG ELEMENT UPDATE -----")
    print(f"Slide         : {slide_num}")
    print(f"Shape Name    : {shape_name}")
    print(f"Element Type  : {element_type}")
    print(f"Static Text   : {static_text}")
    print(f"Data Empty    : {df.empty}")
    print(f"Rows x Cols   : {df.shape}")
    print("--------------------------------")
 
    ok, chart_xml_name = update_element(
        prs, slide_num, shape_name, element_type,
        df, static_text, show_last_label
    )
 
    print(f"Update Success: {ok}")
    if chart_xml_name:
        print(f"Chart XML Ref : {chart_xml_name}")
    print("--------------------------------\n")
 
    return ok, chart_xml_name

In [0]:
# ============================================================
# PATCHED: Save highlights via /tmp (Volumes FUSE limitation)
# ============================================================
def run_deck_update_with_highlights(config_path, template_path, output_path,
                                    max_parallel_queries=15,
                                    update_embedded_excel=True):
    chart_data_map = run_deck_update(
        config_path, template_path, output_path,
        max_parallel_queries, update_embedded_excel
    )

    print(f"\n{'='*60}\nREADING HIGHLIGHTS FROM DB\n{'='*60}")
    highlight_df = read_highlights_from_db()

    if highlight_df.empty:
        print("No highlights found in DB \u2014 skipping")
        return

    prs = Presentation(output_path)
    print(f"\nChart data map: {len(chart_data_map)} charts")
    apply_all_highlights(prs, highlight_df, chart_data_map)

    # Save to local temp first, then copy bytes to Volumes
    local_hl_temp = "/tmp/_deck_highlights.pptx"
    prs.save(local_hl_temp)
    with open(local_hl_temp, 'rb') as f:
        pptx_bytes = f.read()
    with open(output_path, 'wb') as f:
        f.write(pptx_bytes)
    os.remove(local_hl_temp)
    print(f"\nFinal deck saved: {output_path}")

In [0]:
# ============================================================
# 11. ENTRY POINT
# ============================================================

output = run_deck_update(
    config_path=VOLUME_CONFIG_PATH,
    template_path=VOLUME_TEMPLATE_PATH,
    output_path=VOLUME_OUTPUT_PATH,
)

DECK UPDATE STARTED
Config loaded: 1038 elements to update
      slide_num                                                                        shape_name                element_type
0             1                                                                              date                     textbox
1             3                                                                           Shape 6                exec_summary
2             3                                                                               com                exec_summary
3             3                                                                          Shape 16                exec_summary
4             4                                                                          Shape 11                exec_summary
5             4                                                                             payer                exec_summary
6             4                                            

     ERROR fetching SQL_713: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pt_batch_id` cannot be resolved. Did you mean one of the following? [`EPI_Factor`, `BUY_DOWN`, `CHANNEL`, `CLAIM_ID`, `LOT`]. SQLSTATE: 42703; line 7 pos 10;
'Sort ['sort_order ASC NULLS FIRST], true
+- 'Project ['label AS Category#3112408, 'slow_progressor AS Slow Progressor#3112409, 'fast_progressor AS Fast Progressor#3112410]
   +- 'SubqueryAlias combined
      +- 'SubqueryAlias combined
         +- 'Union false, false
            :- 'Union false, false
            :  :- 'Project [(('cohort_order * 3) - 2) AS sort_order#3112441, 'cohort_label AS label#3112442, 'slow_pct AS slow_progressor#3112443, 'fast_pct AS fast_progressor#3112444]
            :  :  +- 'SubqueryAlias p12m_stats
            :  :     +- 'SubqueryAlias p12m_stats
            :  :        +- 'Project ['cohort_order, 'period, 'cohort_label, 'slow_pct, 'fast_pct]
            :  :           +- 'SubqueryAl

/databricks/python_shell/lib/dbruntime/autoreload/discoverability/autoreload_discoverability_hook.py:72: UserWarning: Workbook contains no stylesheet, using openpyxl's defaults
  return orig_warn(*args, **kwargs)



Success: 948, Failed: 90, Saved to: /Volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/narrative_deck_output_path/NARRATIVE_DECK_UPDATED_2026-11-18.pptx


In [0]:
# # Copy output to demo volume.
# # The target volume name is literally `deck_output_path(demo)` with the parentheses,
# # so the replace string must include them exactly.
# dbutils.fs.cp(
#     VOLUME_OUTPUT_PATH,
#     VOLUME_OUTPUT_PATH.replace('/narrative_deck_output_path/', '/deck_output_path(demo)/'),
#     recurse=True
# )

In [0]:
# Write the latest deck info to the state table so Genie can serve it instantly
file_name = os.path.basename(VOLUME_OUTPUT_PATH)
location_url = "https://dbc-5c6e6e7d-7beb.cloud.databricks.com/explore/data/volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/narrative_deck_output_path"

spark.sql("""
    CREATE TABLE IF NOT EXISTS `commercial-us-apo-onc-iia-dev`.infigen_adhocs.narrative_deck_latest (
        location_url STRING,
        file_name    STRING,
        updated_at   TIMESTAMP
    )
""")

spark.sql(f"""
    INSERT OVERWRITE `commercial-us-apo-onc-iia-dev`.infigen_adhocs.narrative_deck_latest
    VALUES ('{location_url}', '{file_name}', current_timestamp())
""")
print(f"\u2705 State table updated: {file_name}")

✅ State table updated: NARRATIVE_DECK_UPDATED_2026-11-18.pptx


In [0]:
# Verify output exists on Volumes
import os
if os.path.exists(VOLUME_OUTPUT_PATH):
    size_mb = os.path.getsize(VOLUME_OUTPUT_PATH) / (1024 * 1024)
    print(f"Deck saved to Volumes: {VOLUME_OUTPUT_PATH} ({size_mb:.2f} MB)")
else:
    print(f"WARNING: Output file not found at {VOLUME_OUTPUT_PATH}")

Deck saved to Volumes: /Volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/narrative_deck_output_path/NARRATIVE_DECK_UPDATED_2026-11-18.pptx (11.47 MB)


In [0]:
import json

result = json.dumps({"deck_path": VOLUME_OUTPUT_PATH, "status": "success"})
dbutils.notebook.exit(result)

# demo

In [0]:
# %sql
# -- Create or refresh the function with a strong comment so Genie's planner picks it up
# CREATE OR REPLACE FUNCTION
#   `commercial-us-apo-onc-iia-dev`.`infigen_adhocs`.`get_deck_refresh_location` ()
# RETURNS STRING
# COMMENT 'Returns the Unity Catalog volume path for the refreshed demo deck. Call this
#          function when a user asks to refresh the deck, download the deck, find the deck,
#          get the deck path, get the deck link, or any similar question about where the
#          generated demo deck lives.'
# RETURN 'https://dbc-5c6e6e7d-7beb.cloud.databricks.com/explore/data/volumes/commercial-us-apo-onc-iia-dev/infigen_adhocs/deck_output_path(demo)?o=4447352726644559';
 
# -- Verify
# SELECT `commercial-us-apo-onc-iia-dev`.`infigen_adhocs`.`get_deck_refresh_location`() AS deck_path;

In [0]:
%sql
-- Sanity check: call the function and confirm it returns the expected path
-- Verify it works
SELECT `commercial-us-apo-onc-iia-dev`.`infigen_adhocs`.`get_deck_refresh_location`() AS deck_path;